# Module 7: EIS Site & Asset Intelligence (with EIS Flex)

This notebook adds a second business unit to the EPOWER demo: **EPOWER Infrastructure Solutions (EIS)**. EIS is a fictional B2B energy-infrastructure business in Germany. It finances, builds and operates **CHP plants, large heat pumps, boilers and thermal storage** at customer sites (hospitals, industrial parks, housing estates, district heating networks), under long-term contracts with **availability and efficiency guarantees**.

**The story:** unified OT and IT data → AI grounded in governed data → measurable € impact.

| Moment | Question to the agent | What makes it work |
|--------|-----------------------|--------------------|
| 1. Risk & root cause | *Which contracts are at risk, what is our € exposure, and why?* → *What should the technician do on Klinikum CHP-2?* | Telemetry + contract guarantees + OEM manual (RAG) in one conversation |
| 2. EIS Flex value | *How much did EIS Flex save, and what are we missing on sites that are not enrolled?* | Real EPEX day-ahead prices → realised vs. missed € |
| 3. Tomorrow (short closer) | *When should the Hamburg-Altona heat pumps run tomorrow?* | Next-day prices → hourly heat pump schedule |

**EIS Flex** applies the Virtual Power Plant principle from Module 1 to large heat pumps. The heat pump produces the day's heat in the **cheapest hours of the day** (real EPEX day-ahead prices) and fills the thermal storage, which supplies the heat in the expensive hours. *Same heat, bought at the cheapest hours.* Heat demand is always met. See [README-module7.md](README-module7.md) for the full concept.

### What this notebook builds
All objects go into **one new schema, `EPOWER_DEMO.EIS_GOLD`**. There are no new dbt models and no new tasks. The only thing reused from Module 1 is the real day-ahead price table `EPOWER_GOLD.MART_DAY_AHEAD_PRICES` (read-only).

| Section | Objects |
|---------|---------|
| 1–2 | Session, prerequisite check, schema, stage, technical parameters |
| 3 | Master data: `EIS_SITES`, `EIS_ASSETS`, `EIS_CONTRACTS` |
| 4 | Operations: `EIS_FAULT_EVENTS`, `EIS_SITE_DISPATCH_HOURLY` (EIS Flex), `EIS_ASSET_TELEMETRY`, `EIS_WORK_ORDERS` |
| 5 | Documents: parse, `AI_EXTRACT` guarantee clauses, Cortex Search `SEARCH_EIS_DOCS` |
| 6 | Business views: asset health, guarantee status, flex margin, heat pump schedule |
| 7 | Semantic view `EIS_OPERATIONS_SEMANTIC_VIEW` |
| 8 | Agent `EIS_OPERATIONS_AGENT` + CoWork registration |
| 9 | Verification and story check |

**Prerequisite:** Module 1 (`01-agentic-ai-foundation/epower_hol_main.ipynb`) has been run. **Runtime:** about 5–10 minutes.

> All data is synthetic and **deterministic**: pseudo-random values come from `HASH()` instead of `RANDOM()`, so every run produces the same story. Only the electricity prices are real.

## 1. Session Setup & Prerequisite Check

Use the role and warehouse from Module 1. The check makes sure the real day-ahead prices are available. EIS Flex needs about 60 days of price history.

In [ ]:
# Get Snowpark session and switch to the demo role/warehouse created in Module 1
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE EPOWER_ROLE").collect()
session.sql("USE WAREHOUSE EPOWER_COMPUTE").collect()
session.sql("USE DATABASE EPOWER_DEMO").collect()
print(f"Connected as: {session.get_current_user()} | role: {session.get_current_role()}")

# Prerequisite: Module 1 price data (real EPEX day-ahead prices)
try:
    days = session.sql("""
        SELECT COUNT(DISTINCT CONVERT_TIMEZONE('UTC', 'Europe/Berlin', START_TIME)::DATE) AS days
        FROM EPOWER_DEMO.EPOWER_GOLD.MART_DAY_AHEAD_PRICES
    """).collect()[0]["DAYS"]
except Exception as e:
    raise RuntimeError("EPOWER_GOLD.MART_DAY_AHEAD_PRICES not found - please run Module 1 first.") from e

if days < 55:
    raise RuntimeError(f"Only {days} days of day-ahead prices found; at least 55 are needed. "
                       "Run CALL EPOWER_DEMO.EPOWER_OPS.BACKFILL_DAY_AHEAD_PRICES() first.")
print(f"Prerequisite OK: {days} days of real day-ahead prices available")

## 2. Schema, Stage & Technical Parameters

**One schema per business unit.** EIS gets its own schema `EIS_GOLD` next to `EPOWER_GOLD`. This keeps ownership, access control and cleanup simple (`DROP SCHEMA`), while both units share the same platform and the same market data.

The **technical parameters** (efficiencies) live in a small table instead of being hard-coded. All calculations in this notebook read them, so you can change an assumption and re-run.

| Parameter | Value | Used for |
|-----------|-------|----------|
| Heat pump COP | 3.2 | 1 MWh heat → 0.31 MWh electricity. **The only parameter EIS Flex needs** |
| CHP efficiency | 38 % electrical, 50 % thermal | Electricity and gas readings of CHP units |
| Boiler efficiency | 92 % | Gas readings of boilers |

In [ ]:
%%sql
-- Schema for the EIS business unit
CREATE SCHEMA IF NOT EXISTS EPOWER_DEMO.EIS_GOLD
    COMMENT = 'EPOWER Infrastructure Solutions (Module 7): sites, assets, contracts, telemetry, EIS Flex';

-- Encrypted stage with directory table for contract documents and OEM manuals
CREATE OR REPLACE STAGE EPOWER_DEMO.EIS_GOLD.EIS_DOCS_STAGE
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
    DIRECTORY = (ENABLE = TRUE);

-- Technical parameters used by all calculations
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_PARAMETERS (
    hp_cop                 FLOAT COMMENT 'Heat pump coefficient of performance (heat out / electricity in)',
    chp_el_eff             FLOAT COMMENT 'CHP electrical efficiency',
    chp_th_eff             FLOAT COMMENT 'CHP thermal efficiency',
    boiler_eff             FLOAT COMMENT 'Gas boiler efficiency'
) AS SELECT 3.2, 0.38, 0.50, 0.92;

SELECT * FROM EPOWER_DEMO.EIS_GOLD.EIS_PARAMETERS;

## 3. Master Data: Sites, Assets, Contracts

In a real EIS landscape this data comes from different systems: the site and customer master from CRM/ERP, the asset register from the asset management system (e.g. SAP PM), and the contract terms from the contract repository. Here we generate it.

**Five "hero" sites carry the demo story.** The other 115 sites make the fleet realistic.

| Site | Type | EIS Flex | Role in the story |
|------|------|----------|-------------------|
| S001 Klinikum Hamburg-Nord | Hospital | No | CHP-2 is degrading → availability guarantee at risk |
| S002 Chemiepark Leuna | Industrial park | Yes | Heat pump outage → guarantee on watch |
| S003 Wärmenetz Hamburg-Altona | District heating | Yes | Large 400 MWh storage → top flex margin |
| S004 Wohnquartier München-Riem | Housing | No | Has heat pump + storage, but storage sits idle → top EIS Flex onboarding candidate |
| S005 Campus Universität Kiel | Campus | No | CHP outage → guarantee at risk |

About 60 % of the generated sites are enrolled in **EIS Flex**. Flex sites get thermal storage sized for ~6 hours of peak demand, baseline sites for ~3 hours.

**Deterministic randomness:** `MOD(ABS(HASH(site_id, 'salt')), 1000) / 1000.0` returns the same "random" number between 0 and 1 for the same input on every run. The salt makes values for different attributes independent of each other.

In [ ]:
%%sql
-- EIS_SITES: 5 hero sites + 115 generated sites across Germany
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_SITES
    COMMENT = 'EIS customer sites (energy centres, district heating networks)'
AS
WITH hero AS (
    SELECT * FROM VALUES
      ('S001','Klinikum Hamburg-Nord','HOSPITAL','Klinikum Hamburg-Nord gGmbH','Hamburg','Hamburg',53.6290,10.0130,FALSE,3.0,20.0),
      ('S002','Chemiepark Leuna Prozesswaerme','INDUSTRIAL_PARK','Chemiepark Leuna Infrastruktur GmbH','Leuna','Sachsen-Anhalt',51.3170,12.0170,TRUE,6.0,60.0),
      ('S003','Waermenetz Hamburg-Altona','DISTRICT_HEATING','Stadtwerke-Quartiersgesellschaft Hamburg-Altona mbH','Hamburg','Hamburg',53.5500,9.9350,TRUE,10.0,400.0),
      ('S004','Wohnquartier Muenchen-Riem','HOUSING','Wohnbau Riem eG','Muenchen','Bayern',48.1330,11.6900,FALSE,5.0,30.0),
      ('S005','Campus Universitaet Kiel','CAMPUS','Christian-Albrechts-Universitaet zu Kiel','Kiel','Schleswig-Holstein',54.3470,10.1130,FALSE,3.0,12.0)
    AS t(site_id, site_name, site_type, customer_name, city, state, lat, lon, flex_enabled, heat_demand_peak_mw, storage_capacity_mwh)
),
cities AS (
    SELECT * FROM VALUES
      (0,'Berlin','Berlin',52.52,13.40),(1,'Hamburg','Hamburg',53.55,10.00),(2,'Muenchen','Bayern',48.14,11.58),
      (3,'Koeln','Nordrhein-Westfalen',50.94,6.96),(4,'Frankfurt am Main','Hessen',50.11,8.68),(5,'Stuttgart','Baden-Wuerttemberg',48.78,9.18),
      (6,'Duesseldorf','Nordrhein-Westfalen',51.23,6.78),(7,'Leipzig','Sachsen',51.34,12.37),(8,'Dortmund','Nordrhein-Westfalen',51.51,7.47),
      (9,'Essen','Nordrhein-Westfalen',51.46,7.01),(10,'Bremen','Bremen',53.08,8.80),(11,'Dresden','Sachsen',51.05,13.74),
      (12,'Hannover','Niedersachsen',52.37,9.73),(13,'Nuernberg','Bayern',49.45,11.08),(14,'Regensburg','Bayern',49.01,12.10),
      (15,'Magdeburg','Sachsen-Anhalt',52.12,11.63),(16,'Rostock','Mecklenburg-Vorpommern',54.09,12.10),(17,'Kassel','Hessen',51.31,9.48),
      (18,'Mannheim','Baden-Wuerttemberg',49.49,8.47),(19,'Erfurt','Thueringen',50.98,11.03)
    AS c(city_idx, city, state, lat, lon)
),
types AS (
    SELECT * FROM VALUES
      (0,'HOSPITAL','Klinikum',3.0),(1,'INDUSTRIAL_PARK','Industriepark',5.0),(2,'HOUSING','Wohnquartier',2.5),
      (3,'DISTRICT_HEATING','Waermenetz',8.0),(4,'CAMPUS','Campus',2.5),(5,'HOUSING','Quartier',2.5),(6,'INDUSTRIAL_PARK','Werk',5.0)
    AS ty(type_idx, site_type, prefix, base_peak_mw)
),
generated AS (
    SELECT 'S' || LPAD((SEQ4() + 6)::VARCHAR, 3, '0') AS site_id
    FROM TABLE(GENERATOR(ROWCOUNT => 115))
),
generated_sites AS (
    SELECT g.site_id,
           ty.prefix || ' ' || c.city || ' ' || g.site_id AS site_name,
           ty.site_type,
           ty.prefix || ' ' || c.city || ' Betriebs GmbH' AS customer_name,
           c.city, c.state,
           ROUND(c.lat + (MOD(ABS(HASH(g.site_id, 'lat')), 1000) / 1000.0 - 0.5) * 0.12, 4) AS lat,
           ROUND(c.lon + (MOD(ABS(HASH(g.site_id, 'lon')), 1000) / 1000.0 - 0.5) * 0.18, 4) AS lon,
           MOD(ABS(HASH(g.site_id, 'flex')), 100) < 60 AS flex_enabled,
           ROUND(ty.base_peak_mw * (0.6 + MOD(ABS(HASH(g.site_id, 'demand')), 1000) / 1000.0 * 0.8), 1) AS heat_demand_peak_mw
    FROM generated g
    JOIN cities c ON c.city_idx = MOD(ABS(HASH(g.site_id, 'city')), 20)
    JOIN types ty ON ty.type_idx = MOD(ABS(HASH(g.site_id, 'type')), 7)
)
SELECT site_id, site_name, site_type, customer_name, city, state, lat, lon, flex_enabled,
       heat_demand_peak_mw, storage_capacity_mwh
FROM hero
UNION ALL
SELECT site_id, site_name, site_type, customer_name, city, state, lat, lon, flex_enabled,
       heat_demand_peak_mw,
       ROUND(heat_demand_peak_mw * IFF(flex_enabled, 6, 3), 0) AS storage_capacity_mwh
FROM generated_sites;

SELECT site_type, COUNT(*) AS sites, SUM(IFF(flex_enabled, 1, 0)) AS flex_sites,
       ROUND(SUM(heat_demand_peak_mw), 1) AS peak_demand_mw
FROM EPOWER_DEMO.EIS_GOLD.EIS_SITES GROUP BY site_type ORDER BY sites DESC;

In [ ]:
%%sql
-- EIS_ASSETS: asset register (CHP, heat pumps, boilers, thermal storage)
-- Hero sites have a fixed configuration; generated sites get a configuration derived from site type, size and Flex status
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_ASSETS
    COMMENT = 'EIS asset register: generation and storage units per site'
AS
WITH hero_assets AS (
    SELECT * FROM VALUES
      ('S001','CHP-1','CHP','EC400',2019), ('S001','CHP-2','CHP','EC400',2019),
      ('S001','BOILER-1','BOILER','B2500',2019), ('S001','BOILER-2','BOILER','B2500',2019), ('S001','TES-1','THERMAL_STORAGE','TES',2019),
      ('S002','CHP-1','CHP','EC2000',2021), ('S002','HP-1','HEAT_PUMP','HP2000',2021), ('S002','HP-2','HEAT_PUMP','HP2000',2021),
      ('S002','BOILER-1','BOILER','B5000',2021), ('S002','TES-1','THERMAL_STORAGE','TES',2021),
      ('S003','CHP-1','CHP','EC2000',2020), ('S003','CHP-2','CHP','EC2000',2020),
      ('S003','HP-1','HEAT_PUMP','HP2000',2020), ('S003','HP-2','HEAT_PUMP','HP2000',2020), ('S003','HP-3','HEAT_PUMP','HP2000',2020),
      ('S003','BOILER-1','BOILER','B6000',2020), ('S003','BOILER-2','BOILER','B6000',2020), ('S003','TES-1','THERMAL_STORAGE','TES',2020),
      ('S004','CHP-1','CHP','EC400',2016), ('S004','HP-1','HEAT_PUMP','HP2000',2022),
      ('S004','BOILER-1','BOILER','B3000',2016), ('S004','BOILER-2','BOILER','B3000',2016), ('S004','TES-1','THERMAL_STORAGE','TES',2016),
      ('S005','CHP-1','CHP','EC400',2018), ('S005','CHP-2','CHP','EC400',2018),
      ('S005','BOILER-1','BOILER','B2500',2018), ('S005','TES-1','THERMAL_STORAGE','TES',2018)
    AS h(site_id, asset_code, asset_type, model, commissioning_year)
),
slots AS (
    SELECT * FROM VALUES ('CHP',1),('CHP',2),('HEAT_PUMP',1),('HEAT_PUMP',2),('BOILER',1),('BOILER',2),('THERMAL_STORAGE',1)
    AS s(asset_type, slot_no)
),
generated_assets AS (
    SELECT st.site_id,
           CASE sl.asset_type WHEN 'CHP' THEN 'CHP' WHEN 'HEAT_PUMP' THEN 'HP' WHEN 'BOILER' THEN 'BOILER' ELSE 'TES' END
               || '-' || sl.slot_no AS asset_code,
           sl.asset_type,
           CASE sl.asset_type
               WHEN 'CHP' THEN IFF(st.site_type IN ('DISTRICT_HEATING','INDUSTRIAL_PARK'), 'EC2000', 'EC400')
               WHEN 'HEAT_PUMP' THEN IFF(st.heat_demand_peak_mw >= 4, 'HP2000', 'HP1000')
               WHEN 'BOILER' THEN 'B' || ROUND(st.heat_demand_peak_mw * 800)::VARCHAR
               ELSE 'TES' END AS model,
           2012 + MOD(ABS(HASH(st.site_id, 'year')), 12) AS commissioning_year,
           CASE sl.asset_type
               WHEN 'CHP' THEN 1 + MOD(ABS(HASH(st.site_id, 'n_chp')), 2)
               WHEN 'HEAT_PUMP' THEN IFF(st.flex_enabled, 1, 0) + MOD(ABS(HASH(st.site_id, 'n_hp')), 2)
               WHEN 'BOILER' THEN IFF(st.heat_demand_peak_mw > 5, 2, 1)
               ELSE 1 END AS units_at_site
    FROM EPOWER_DEMO.EIS_GOLD.EIS_SITES st
    CROSS JOIN slots sl
    WHERE st.site_id > 'S005'
),
all_assets AS (
    SELECT site_id, asset_code, asset_type, model, commissioning_year FROM hero_assets
    UNION ALL
    SELECT site_id, asset_code, asset_type, model, commissioning_year FROM generated_assets
    WHERE SPLIT_PART(asset_code, '-', 2)::INT <= units_at_site
)
SELECT a.site_id || '-' || a.asset_code AS asset_id,
       a.site_id,
       a.asset_code,
       a.asset_type,
       CASE a.asset_type WHEN 'CHP' THEN 'EcoPower Energietechnik' WHEN 'HEAT_PUMP' THEN 'ThermaFlux Industrial Heat'
                         WHEN 'BOILER' THEN 'Kesselbau Nord' ELSE 'Speichertechnik Mitte' END AS oem,
       CASE a.asset_type WHEN 'CHP' THEN 'EcoPower ' WHEN 'HEAT_PUMP' THEN 'ThermaFlux ' WHEN 'BOILER' THEN 'Kessel ' ELSE 'Waermespeicher ' END
           || a.model AS model,
       CASE a.model WHEN 'EC400' THEN 0.40 WHEN 'EC2000' THEN 2.00 ELSE 0 END AS el_capacity_mw,
       CASE WHEN a.model = 'EC400' THEN 0.52
            WHEN a.model = 'EC2000' THEN 2.60
            WHEN a.model = 'HP2000' THEN 2.00
            WHEN a.model = 'HP1000' THEN 1.00
            WHEN a.asset_type = 'BOILER' THEN TRY_TO_NUMBER(SUBSTR(a.model, 2)) / 1000.0
            ELSE 0 END AS th_capacity_mw,
       IFF(a.asset_type = 'THERMAL_STORAGE', s.storage_capacity_mwh, 0) AS storage_capacity_mwh,
       CASE a.asset_type WHEN 'CHP' THEN 88 WHEN 'HEAT_PUMP' THEN 320 WHEN 'BOILER' THEN 92 END AS nominal_efficiency_pct,
       a.commissioning_year
FROM all_assets a
JOIN EPOWER_DEMO.EIS_GOLD.EIS_SITES s ON s.site_id = a.site_id;

SELECT asset_type, COUNT(*) AS assets, ROUND(SUM(th_capacity_mw), 1) AS th_capacity_mw,
       ROUND(SUM(el_capacity_mw), 1) AS el_capacity_mw, ROUND(SUM(storage_capacity_mwh)) AS storage_mwh
FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSETS GROUP BY asset_type ORDER BY assets DESC;

In [ ]:
%%sql
-- EIS_CONTRACTS: one long-term contract per site with performance guarantees
-- Hero contracts (S001-S003) mirror the contract documents in docs/; in section 5, AI_EXTRACT reads
-- the guarantee clauses from the documents and checks them against this table.
--   availability_basis: PER_UNIT = every unit must meet the guarantee | AVERAGE = average over units | SUPPLY = heat supply level
--   availability_scope: which asset types count (CHP | CHP_HP)
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_CONTRACTS
    COMMENT = 'EIS customer contracts with availability/efficiency guarantees, penalties and EIS Flex margin share'
AS
WITH hero AS (
    SELECT * FROM VALUES
      ('EIS-C-0001','S001','CONTRACTING','2019-01-01','2034-12-31',1850000,97.0,'PER_UNIT','CHP',85.0,'CHP',45000,30000,NULL),
      ('EIS-C-0002','S002','EAAS','2021-04-01','2036-03-31',2400000,96.0,'AVERAGE','CHP_HP',300.0,'HEAT_PUMP',25000,2000,60),
      ('EIS-C-0003','S003','HEAT_SUPPLY','2020-10-01','2040-09-30',3200000,99.5,'SUPPLY','CHP_HP',85.0,'CHP',60000,25000,70),
      ('EIS-C-0004','S004','HEAT_SUPPLY','2016-01-01','2031-12-31',950000,97.0,'AVERAGE','CHP_HP',85.0,'CHP',20000,15000,NULL),
      ('EIS-C-0005','S005','CONTRACTING','2018-07-01','2033-06-30',1100000,97.0,'PER_UNIT','CHP',85.0,'CHP',20000,15000,NULL)
    AS h(contract_id, site_id, contract_type, start_date, end_date, annual_fee_eur,
         guaranteed_availability_pct, availability_basis, availability_scope,
         guaranteed_efficiency_pct, efficiency_scope, penalty_availability_eur_per_pt, penalty_efficiency_eur_per_pt, flex_share_eis_pct)
)
SELECT contract_id, site_id, contract_type, start_date::DATE AS start_date, end_date::DATE AS end_date, annual_fee_eur,
       guaranteed_availability_pct, availability_basis, availability_scope,
       guaranteed_efficiency_pct, efficiency_scope, penalty_availability_eur_per_pt, penalty_efficiency_eur_per_pt,
       flex_share_eis_pct, 'GENERATED' AS terms_source
FROM hero
UNION ALL
SELECT 'EIS-C-' || LPAD(SUBSTR(s.site_id, 2), 4, '0') AS contract_id,
       s.site_id,
       CASE MOD(ABS(HASH(s.site_id, 'ctype')), 3) WHEN 0 THEN 'CONTRACTING' WHEN 1 THEN 'EAAS' ELSE 'HEAT_SUPPLY' END,
       DATEADD(MONTH, -MOD(ABS(HASH(s.site_id, 'start')), 120), DATE_TRUNC('MONTH', CURRENT_DATE())) AS start_date,
       DATEADD(YEAR, 15, DATEADD(MONTH, -MOD(ABS(HASH(s.site_id, 'start')), 120), DATE_TRUNC('MONTH', CURRENT_DATE()))) AS end_date,
       ROUND(s.heat_demand_peak_mw * (220000 + MOD(ABS(HASH(s.site_id, 'fee')), 1000) * 100), -3) AS annual_fee_eur,
       96.0 + MOD(ABS(HASH(s.site_id, 'avail')), 3) * 0.5 AS guaranteed_availability_pct,
       'AVERAGE', 'CHP_HP',
       85.0, 'CHP',
       ROUND(s.heat_demand_peak_mw * 6000, -3) AS penalty_availability_eur_per_pt,
       ROUND(s.heat_demand_peak_mw * 4000, -3) AS penalty_efficiency_eur_per_pt,
       IFF(s.flex_enabled, IFF(MOD(ABS(HASH(s.site_id, 'share')), 2) = 0, 60, 70), NULL) AS flex_share_eis_pct,
       'GENERATED'
FROM EPOWER_DEMO.EIS_GOLD.EIS_SITES s
WHERE s.site_id > 'S005';

SELECT contract_type, COUNT(*) AS contracts, ROUND(SUM(annual_fee_eur) / 1e6, 1) AS annual_fees_meur
FROM EPOWER_DEMO.EIS_GOLD.EIS_CONTRACTS GROUP BY contract_type;

## 4. Operations: Faults, EIS Flex Dispatch, Telemetry, Work Orders

This section simulates **60 days of hourly plant operation**: what an OT historian or SCADA system would deliver.

### 4a. Fault events (the planted story)
Most assets run with rare, short random faults. Three events are planted on purpose so the demo story is reproducible:

| Asset | Fault | Effect |
|-------|-------|--------|
| S001-CHP-2 (Klinikum) | 3 × **F-217** exhaust heat exchanger fouling (43 h in the last 30 days) + efficiency drifting down ~0.15 pp/day | Availability ~94 % vs. 97 % guarantee |
| S002-HP-1 (Leuna) | **H-512** frequency converter, 72 h outage | Average availability just above guarantee → *watch* |
| S005-CHP-1 (Kiel) | **F-142** low lube oil pressure, 26 h outage | Availability ~96.4 % vs. 97 % guarantee |

All timestamps are relative to `CURRENT_DATE()`, so the story stays current whenever you run the notebook.

In [ ]:
%%sql
-- EIS_FAULT_EVENTS: unplanned outages (planted story events + random background faults)
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_FAULT_EVENTS
    COMMENT = 'Unplanned asset outages with fault codes (as reported by plant control / SCADA)'
AS
WITH planted AS (
    SELECT * FROM VALUES
      ('S001-CHP-2', 'F-217', -24, 10, 12),
      ('S001-CHP-2', 'F-217', -15, 14, 14),
      ('S001-CHP-2', 'F-217',  -6,  2, 17),
      ('S002-HP-1',  'H-512', -16,  6, 72),
      ('S005-CHP-1', 'F-142', -11,  8, 26)
    AS p(asset_id, fault_code, day_offset, start_hour, duration_h)
),
random_faults AS (
    SELECT a.asset_id,
           CASE a.asset_type
               WHEN 'CHP' THEN ARRAY_CONSTRUCT('F-101','F-142','F-305','F-410')[MOD(ABS(HASH(a.asset_id, 'code')), 4)]::VARCHAR
               WHEN 'HEAT_PUMP' THEN ARRAY_CONSTRUCT('H-110','H-220','H-305')[MOD(ABS(HASH(a.asset_id, 'code')), 3)]::VARCHAR
               ELSE 'K-01' END AS fault_code,
           -(1 + MOD(ABS(HASH(a.asset_id, 'day')), 59)) AS day_offset,
           MOD(ABS(HASH(a.asset_id, 'hour')), 24) AS start_hour,
           2 + MOD(ABS(HASH(a.asset_id, 'dur')), 7) AS duration_h
    FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSETS a
    WHERE a.asset_type IN ('CHP', 'HEAT_PUMP', 'BOILER')
      AND a.site_id > 'S005'                                   -- hero sites only carry planted events
      AND MOD(ABS(HASH(a.asset_id, 'fault')), 100) < 20       -- ~20 % of assets had one short fault
),
events AS (
    SELECT asset_id, fault_code, day_offset, start_hour, duration_h, TRUE AS is_planted FROM planted
    UNION ALL
    SELECT asset_id, fault_code, day_offset, start_hour, duration_h, FALSE FROM random_faults
)
SELECT 'FE-' || LPAD(ROW_NUMBER() OVER (ORDER BY e.asset_id, e.day_offset)::VARCHAR, 4, '0') AS event_id,
       e.asset_id,
       a.site_id,
       e.fault_code,
       CASE e.fault_code
           WHEN 'F-101' THEN 'Generator overcurrent'
           WHEN 'F-142' THEN 'Low lube oil pressure'
           WHEN 'F-217' THEN 'Exhaust heat exchanger fouling - high exhaust temperature'
           WHEN 'F-305' THEN 'Cooling water over-temperature'
           WHEN 'F-410' THEN 'Ignition misfire'
           WHEN 'H-110' THEN 'High pressure trip'
           WHEN 'H-220' THEN 'Compressor motor overload'
           WHEN 'H-305' THEN 'Low source temperature'
           WHEN 'H-512' THEN 'Frequency converter fault'
           WHEN 'K-01'  THEN 'Burner flame failure'
       END AS fault_description,
       DATEADD(HOUR, e.start_hour, DATEADD(DAY, e.day_offset, CURRENT_DATE())::TIMESTAMP_NTZ) AS start_ts,
       DATEADD(HOUR, e.start_hour + e.duration_h, DATEADD(DAY, e.day_offset, CURRENT_DATE())::TIMESTAMP_NTZ) AS end_ts,
       e.duration_h,
       e.is_planted
FROM events e
JOIN EPOWER_DEMO.EIS_GOLD.EIS_ASSETS a ON a.asset_id = e.asset_id;

SELECT asset_id, fault_code, fault_description, start_ts, duration_h
FROM EPOWER_DEMO.EIS_GOLD.EIS_FAULT_EVENTS WHERE is_planted ORDER BY start_ts;

### 4b. EIS Flex: same heat, bought at the cheapest hours

This is the heart of the module. Every site covers its hourly heat demand **heat-led**: CHP first, then the heat pump, then the boiler.

**EIS Flex changes only one thing: *when* the heat pump runs.**

1. Per site and day, add up how much heat the heat pump has to deliver.
2. Rank the day's hours by **real EPEX day-ahead price** (from Module 1).
3. Let the heat pump produce that amount at full load in the cheapest hours.
4. The thermal storage takes the difference: it **charges** when the heat pump produces more than needed in that hour, and **discharges** when it produces less.

The storage size limits the shift: if the storage is smaller than the day's heat pump heat, only part of the production moves to the cheapest hours. CHP and boiler run exactly the same with and without EIS Flex, so **heat demand is always met**.

```
flex_margin_eur = heat pump electricity cost (normal) − heat pump electricity cost (EIS Flex)
electricity cost = heat pump heat / COP × day-ahead price of the hour
```

We calculate both variants for **every** site. On **EIS Flex sites** the margin is *realised*. On **baseline sites** it is the margin *missed* by not being in the program, which is the roll-out business case. Units in a fault event have no capacity in that hour.

In [ ]:
%%sql
-- Scale up temporarily for the data generation (scaled down again at the end of section 4)
ALTER WAREHOUSE EPOWER_COMPUTE SET WAREHOUSE_SIZE = 'MEDIUM';

-- EIS_SITE_DISPATCH_HOURLY: heat demand, real day-ahead price, heat-led operation and EIS Flex heat pump schedule per site and hour
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_SITE_DISPATCH_HOURLY
    COMMENT = 'Hourly site operation: heat demand, day-ahead price, heat-led vs. EIS Flex heat pump schedule, storage, electricity cost and flex margin'
AS
WITH params AS (
    SELECT hp_cop FROM EPOWER_DEMO.EIS_GOLD.EIS_PARAMETERS
),
hours AS (
    -- 60 full days (CET local time), ending yesterday 23:00
    SELECT DATEADD(HOUR, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, DATEADD(DAY, -60, CURRENT_DATE())::TIMESTAMP_NTZ) AS ts
    FROM TABLE(GENERATOR(ROWCOUNT => 1440))
),
prices AS (
    -- Module 1 prices are 15-min intervals in UTC -> hourly averages in CET
    SELECT DATE_TRUNC('HOUR', CONVERT_TIMEZONE('UTC', 'Europe/Berlin', start_time)) AS ts,
           AVG(price_eur_mwh) AS price_eur_mwh
    FROM EPOWER_DEMO.EPOWER_GOLD.MART_DAY_AHEAD_PRICES
    GROUP BY 1
),
capacity AS (
    -- Available thermal capacity per site and hour (units in a fault event are unavailable)
    SELECT a.site_id, h.ts,
           SUM(IFF(a.asset_type = 'CHP'       AND f.event_id IS NULL, a.th_capacity_mw, 0)) AS chp_cap,
           SUM(IFF(a.asset_type = 'HEAT_PUMP' AND f.event_id IS NULL, a.th_capacity_mw, 0)) AS hp_cap
    FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSETS a
    CROSS JOIN hours h
    LEFT JOIN EPOWER_DEMO.EIS_GOLD.EIS_FAULT_EVENTS f
           ON f.asset_id = a.asset_id AND h.ts >= f.start_ts AND h.ts < f.end_ts
    WHERE a.asset_type IN ('CHP', 'HEAT_PUMP')
    GROUP BY a.site_id, h.ts
),
demand AS (
    -- Heat demand = peak x season x daily profile x noise (deterministic)
    SELECT s.site_id, s.flex_enabled, s.storage_capacity_mwh, h.ts,
           s.heat_demand_peak_mw
             * (0.55 + 0.35 * COS(2 * PI() * (DAYOFYEAR(h.ts) - 15) / 365))
             * CASE
                 WHEN s.site_type IN ('HOUSING', 'DISTRICT_HEATING')
                   THEN 0.65 + 0.35 * (EXP(-POWER(HOUR(h.ts) - 7, 2) / 4) + EXP(-POWER(HOUR(h.ts) - 19, 2) / 6))
                 WHEN s.site_type = 'HOSPITAL'
                   THEN 0.85 + IFF(HOUR(h.ts) BETWEEN 6 AND 20, 0.15, 0)
                 WHEN s.site_type = 'INDUSTRIAL_PARK'
                   THEN IFF(DAYOFWEEKISO(h.ts) <= 5, IFF(HOUR(h.ts) BETWEEN 6 AND 22, 1.0, 0.85), 0.65)
                 ELSE IFF(DAYOFWEEKISO(h.ts) <= 5 AND HOUR(h.ts) BETWEEN 7 AND 19, 1.0, 0.55)
               END
             * (0.93 + 0.14 * MOD(ABS(HASH(s.site_id, h.ts)), 1000) / 1000.0) AS heat_demand_mwh
    FROM EPOWER_DEMO.EIS_GOLD.EIS_SITES s
    CROSS JOIN hours h
),
heat_led AS (
    -- Normal (heat-led) operation: CHP first, then heat pump, then boiler
    SELECT d.*,
           COALESCE(c.chp_cap, 0) AS chp_cap,
           COALESCE(c.hp_cap, 0) AS hp_cap,
           p.price_eur_mwh IS NOT NULL AS has_market_price,
           COALESCE(p.price_eur_mwh, AVG(p.price_eur_mwh) OVER (PARTITION BY d.site_id, d.ts::DATE)) AS price_eur_mwh,
           LEAST(d.heat_demand_mwh, COALESCE(c.chp_cap, 0)) AS chp_heat,
           LEAST(d.heat_demand_mwh - LEAST(d.heat_demand_mwh, COALESCE(c.chp_cap, 0)), COALESCE(c.hp_cap, 0)) AS hp_heat_normal
    FROM demand d
    LEFT JOIN capacity c ON c.site_id = d.site_id AND c.ts = d.ts
    LEFT JOIN prices p ON p.ts = d.ts
),
daily AS (
    SELECT hl.*,
           SUM(hl.hp_heat_normal) OVER (PARTITION BY hl.site_id, hl.ts::DATE) AS hp_heat_day,
           -- capacity of all cheaper hours of the same day (for "fill the cheapest hours first")
           COALESCE(SUM(hl.hp_cap) OVER (PARTITION BY hl.site_id, hl.ts::DATE ORDER BY hl.price_eur_mwh, hl.ts
                                         ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) AS hp_cap_cheaper_hours,
           RANK() OVER (PARTITION BY hl.site_id, hl.ts::DATE ORDER BY hl.price_eur_mwh) AS price_rank_in_day
    FROM heat_led hl
),
flex AS (
    SELECT dl.*,
           -- Heat pump at full load in the cheapest hours until the day's heat is produced
           LEAST(dl.hp_cap, GREATEST(0, dl.hp_heat_day - dl.hp_cap_cheaper_hours)) AS hp_heat_cheapest,
           -- Share of the day's heat pump heat that the storage can shift (85 % usable capacity)
           IFF(dl.hp_heat_day > 0, LEAST(1, dl.storage_capacity_mwh * 0.85 / dl.hp_heat_day), 0) AS shift_share
    FROM daily dl
),
flex_hp AS (
    SELECT f.*,
           (1 - f.shift_share) * f.hp_heat_normal + f.shift_share * f.hp_heat_cheapest AS hp_heat_flex
    FROM flex f
),
storage AS (
    SELECT fh.*,
           fh.hp_heat_flex - fh.hp_heat_normal AS storage_flow,      -- > 0 charge, < 0 discharge
           SUM(fh.hp_heat_flex - fh.hp_heat_normal) OVER (PARTITION BY fh.site_id, fh.ts::DATE ORDER BY fh.ts) AS storage_level
    FROM flex_hp fh
)
SELECT
    st.ts,
    st.ts::DATE AS day,
    st.site_id,
    st.flex_enabled,
    IFF(st.flex_enabled, 'EIS_FLEX', 'HEAT_LED') AS operating_mode,
    ROUND(st.price_eur_mwh, 2) AS price_eur_mwh,
    st.has_market_price,
    st.price_rank_in_day,
    ROUND(st.heat_demand_mwh, 3) AS heat_demand_mwh,
    st.chp_cap AS chp_capacity_avail_mw,
    st.hp_cap AS hp_capacity_avail_mw,
    -- CHP and boiler: identical in both modes
    ROUND(st.chp_heat, 3) AS chp_heat_mwh,
    ROUND(st.heat_demand_mwh - st.chp_heat - st.hp_heat_normal, 3) AS boiler_heat_mwh,
    -- Heat pump: normal vs. EIS Flex
    ROUND(st.hp_heat_normal, 3) AS hp_heat_normal_mwh,
    ROUND(st.hp_heat_flex, 3) AS hp_heat_flex_mwh,
    ROUND(st.storage_flow, 3) AS flex_storage_flow_mwh,
    -- Storage state of charge: starts the day so that its lowest point is 10 %
    ROUND(10 + 100 * (st.storage_level - LEAST(0, MIN(st.storage_level) OVER (PARTITION BY st.site_id, st.ts::DATE)))
               / NULLIF(st.storage_capacity_mwh, 0), 1) AS flex_storage_soc_pct,
    -- Electricity cost of the heat pump
    ROUND(st.hp_heat_normal / p.hp_cop * st.price_eur_mwh, 2) AS hp_cost_normal_eur,
    ROUND(st.hp_heat_flex / p.hp_cop * st.price_eur_mwh, 2) AS hp_cost_flex_eur,
    ROUND(hp_cost_normal_eur - hp_cost_flex_eur, 2) AS flex_margin_eur,
    -- Actual operation: EIS Flex on enrolled sites, heat-led everywhere else
    IFF(st.flex_enabled, hp_heat_flex_mwh, hp_heat_normal_mwh) AS actual_hp_heat_mwh,
    IFF(st.flex_enabled, flex_storage_flow_mwh, 0) AS actual_storage_flow_mwh,
    IFF(st.flex_enabled, flex_storage_soc_pct, 50) AS actual_storage_soc_pct,
    IFF(st.flex_enabled, hp_cost_flex_eur, hp_cost_normal_eur) AS actual_hp_cost_eur
FROM storage st
CROSS JOIN params p;

-- First look: realised (Flex sites) vs. missed (baseline sites) margin over 60 days
SELECT IFF(flex_enabled, 'EIS Flex sites (realised)', 'Baseline sites (missed)') AS site_group,
       COUNT(DISTINCT site_id) AS sites,
       ROUND(SUM(flex_margin_eur)) AS flex_margin_eur_60d,
       ROUND(SUM(hp_heat_normal_mwh * price_eur_mwh) / NULLIF(SUM(hp_heat_normal_mwh), 0), 1) AS avg_price_normal_eur_mwh,
       ROUND(SUM(hp_heat_flex_mwh * price_eur_mwh) / NULLIF(SUM(hp_heat_flex_mwh), 0), 1) AS avg_price_flex_eur_mwh
FROM EPOWER_DEMO.EIS_GOLD.EIS_SITE_DISPATCH_HOURLY
GROUP BY 1;

### 4c. Asset telemetry (OT data)

The site dispatch is now broken down to **individual assets**, which is the level an OT historian delivers. Each unit gets its share of the site's heat output (by capacity), an efficiency reading, a status and, if applicable, a fault code.

- **Efficiency:** CHP units run around 88 % total efficiency. **S001-CHP-2 drifts down by ~0.15 percentage points per day over the last 30 days**, the typical signature of exhaust heat exchanger fouling (see the OEM manual in section 5). While it runs below 85 %, it occasionally raises **F-217 warnings**.
- **Heat pumps** report efficiency as COP × 100 (320 = COP 3.2).
- **Thermal storage** reports its state of charge (SOC). On baseline sites it sits flat at 50 %, because the storage is unused.

In [ ]:
%%sql
-- EIS_ASSET_TELEMETRY: hourly readings per asset (~600 assets x 1,440 hours)
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_ASSET_TELEMETRY
    COMMENT = 'Hourly OT telemetry per asset: status, heat/electricity output, efficiency, storage SOC, fault codes'
AS
WITH econ AS (
    SELECT chp_el_eff, chp_th_eff, hp_cop FROM EPOWER_DEMO.EIS_GOLD.EIS_PARAMETERS
),
assets AS (
    SELECT a.*,
           SUM(IFF(a.asset_type = 'BOILER', a.th_capacity_mw, 0)) OVER (PARTITION BY a.site_id) AS site_boiler_cap
    FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSETS a
),
base AS (
    SELECT d.ts, d.day, a.asset_id, a.site_id, a.asset_type, a.th_capacity_mw, a.nominal_efficiency_pct,
           d.operating_mode,
           f.fault_code AS trip_code,
           f.event_id IS NOT NULL AS is_fault,
           MOD(ABS(HASH(a.asset_id, d.ts)), 1000) / 1000.0 AS u,
           CASE a.asset_type
               WHEN 'CHP'       THEN d.chp_heat_mwh * a.th_capacity_mw / NULLIF(d.chp_capacity_avail_mw, 0)
               WHEN 'HEAT_PUMP' THEN d.actual_hp_heat_mwh * a.th_capacity_mw / NULLIF(d.hp_capacity_avail_mw, 0)
               WHEN 'BOILER'    THEN d.boiler_heat_mwh * a.th_capacity_mw / NULLIF(a.site_boiler_cap, 0)
               ELSE GREATEST(-d.actual_storage_flow_mwh, 0)          -- storage: heat discharged to the network
           END AS heat_share,
           GREATEST(d.actual_storage_flow_mwh, 0) AS storage_charge,
           d.actual_storage_soc_pct
    FROM EPOWER_DEMO.EIS_GOLD.EIS_SITE_DISPATCH_HOURLY d
    JOIN assets a ON a.site_id = d.site_id
    LEFT JOIN EPOWER_DEMO.EIS_GOLD.EIS_FAULT_EVENTS f
           ON f.asset_id = a.asset_id AND d.ts >= f.start_ts AND d.ts < f.end_ts
),
readings AS (
    SELECT b.*,
           IFF(b.is_fault, 0, COALESCE(b.heat_share, 0)) AS heat_output,
           CASE b.asset_type
               WHEN 'CHP' THEN 86.8 + 1.6 * b.u
                    -- planted degradation: S001-CHP-2 loses ~0.15 pp per day over the last 30 days
                    - IFF(b.asset_id = 'S001-CHP-2', 0.15 * GREATEST(0, DATEDIFF(DAY, DATEADD(DAY, -30, CURRENT_DATE()), b.ts)), 0)
               WHEN 'HEAT_PUMP' THEN 300 + 40 * b.u
               WHEN 'BOILER' THEN 91 + 2 * b.u
           END AS efficiency
    FROM base b
)
SELECT
    r.ts,
    r.day,
    r.asset_id,
    r.site_id,
    r.asset_type,
    CASE WHEN r.is_fault THEN 'FAULT'
         WHEN r.asset_type = 'THERMAL_STORAGE' THEN IFF(r.heat_output > 0.001 OR r.storage_charge > 0.001, 'ACTIVE', 'STANDBY')
         WHEN r.heat_output > 0.001 THEN 'RUNNING'
         ELSE 'STANDBY' END AS status,
    r.operating_mode,
    ROUND(r.heat_output, 3) AS heat_output_mwh,
    ROUND(IFF(r.asset_type = 'CHP', r.heat_output * e.chp_el_eff / e.chp_th_eff, 0), 3) AS el_output_mwh,
    ROUND(IFF(r.asset_type = 'HEAT_PUMP', r.heat_output / (r.efficiency / 100), 0), 3) AS el_consumption_mwh,
    ROUND(CASE r.asset_type
              WHEN 'CHP' THEN r.heat_output / e.chp_th_eff * (88 / r.efficiency)
              WHEN 'BOILER' THEN r.heat_output / (r.efficiency / 100)
              ELSE 0 END, 3) AS gas_consumption_mwh,
    IFF(r.heat_output > 0.001 AND r.asset_type <> 'THERMAL_STORAGE', ROUND(r.efficiency, 2), NULL) AS efficiency_pct,
    IFF(r.asset_type = 'THERMAL_STORAGE', r.actual_storage_soc_pct, NULL) AS storage_soc_pct,
    CASE WHEN r.is_fault THEN r.trip_code
         -- F-217 warnings while the fouled heat exchanger runs below 85 % efficiency
         WHEN r.asset_id = 'S001-CHP-2' AND r.heat_output > 0.001 AND r.efficiency < 85 AND r.u < 0.06 THEN 'F-217'
    END AS fault_code,
    CASE WHEN r.is_fault THEN 'TRIP'
         WHEN r.asset_id = 'S001-CHP-2' AND r.heat_output > 0.001 AND r.efficiency < 85 AND r.u < 0.06 THEN 'WARNING'
    END AS alarm_level
FROM readings r
CROSS JOIN econ e;

SELECT asset_type, status, COUNT(*) AS readings, ROUND(SUM(heat_output_mwh)) AS heat_mwh
FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSET_TELEMETRY GROUP BY 1, 2 ORDER BY 1, 2;

### 4d. Maintenance work orders

In reality this comes from the ERP / field service system. There are two types:
- **Preventive:** routine maintenance on generation units.
- **Corrective:** one per fault event. Note the work orders for **S001-CHP-2**: technicians restarted the unit after each F-217 trip, but nobody cleaned the heat exchanger, and the latest order is still **open without a spare part ordered**. The agent will find this and recommend the procedure from the OEM manual.

In [ ]:
%%sql
-- EIS_WORK_ORDERS: preventive maintenance + one corrective work order per fault event
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_WORK_ORDERS
    COMMENT = 'Maintenance work orders (preventive and corrective) from the field service system'
AS
WITH technicians AS (
    SELECT ARRAY_CONSTRUCT('M. Becker','S. Yilmaz','J. Hoffmann','A. Nowak','L. Schulz','K. Petersen','T. Wagner','N. Richter') AS names
),
preventive AS (
    SELECT a.asset_id, a.site_id, a.asset_type, s.n,
           DATEADD(HOUR, 8 + MOD(ABS(HASH(a.asset_id, s.n, 'h')), 6),
                   DATEADD(DAY, -(1 + MOD(ABS(HASH(a.asset_id, s.n, 'd')), 59)), CURRENT_DATE())::TIMESTAMP_NTZ) AS created_ts
    FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSETS a
    CROSS JOIN (SELECT 1 AS n UNION ALL SELECT 2) s
    WHERE a.asset_type IN ('CHP', 'HEAT_PUMP', 'BOILER')
),
corrective AS (
    SELECT f.*,
           ROW_NUMBER() OVER (PARTITION BY f.asset_id ORDER BY f.start_ts DESC) AS recency
    FROM EPOWER_DEMO.EIS_GOLD.EIS_FAULT_EVENTS f
),
orders AS (
    SELECT p.asset_id, p.site_id, p.created_ts,
           DATEADD(HOUR, 4, p.created_ts) AS closed_ts,
           'PREVENTIVE' AS wo_type, NULL AS fault_code,
           CASE p.asset_type
               WHEN 'CHP' THEN 'Scheduled service: oil and filter change, spark plug inspection'
               WHEN 'HEAT_PUMP' THEN 'Scheduled service: source filter cleaning, refrigerant leak check'
               ELSE 'Scheduled service: burner inspection and flue gas measurement' END AS description,
           'CLOSED' AS status, NULL AS spare_part_kit,
           400 + MOD(ABS(HASH(p.asset_id, p.n, 'cost')), 1200) AS cost_eur,
           MOD(ABS(HASH(p.asset_id, p.n, 'tech')), 8) AS tech_idx
    FROM preventive p
    UNION ALL
    SELECT c.asset_id, c.site_id, c.start_ts,
           CASE WHEN c.asset_id = 'S001-CHP-2' AND c.recency = 1 THEN NULL ELSE c.end_ts END,
           'CORRECTIVE', c.fault_code,
           CASE
               WHEN c.asset_id = 'S001-CHP-2' AND c.recency = 1
                   THEN 'F-217 trip again (3rd time in 3 weeks). High exhaust temperature after heat exchanger. Unit restarted, root cause not resolved. Cleaning to be scheduled - no spare part ordered yet.'
               WHEN c.asset_id = 'S001-CHP-2'
                   THEN 'F-217 trip - high exhaust temperature. Alarm reset and unit restarted. No further action.'
               WHEN c.fault_code = 'H-512'
                   THEN 'H-512 frequency converter failure. Replacement module TF-KIT-512 taken from central warehouse, installed and commissioned.'
               WHEN c.fault_code = 'F-142'
                   THEN 'F-142 low lube oil pressure. Oil pump replaced, oil level corrected, test run OK.'
               ELSE c.fault_description || ' - fault cleared, unit restarted.'
           END,
           CASE WHEN c.asset_id = 'S001-CHP-2' AND c.recency = 1 THEN 'OPEN' ELSE 'CLOSED' END,
           CASE WHEN c.fault_code = 'H-512' THEN 'TF-KIT-512' END,
           CASE WHEN c.fault_code = 'H-512' THEN 18500 WHEN c.fault_code = 'F-142' THEN 6200
                ELSE 300 + MOD(ABS(HASH(c.event_id, 'cost')), 900) END,
           MOD(ABS(HASH(c.event_id, 'tech')), 8)
    FROM corrective c
)
SELECT 'WO-' || LPAD(ROW_NUMBER() OVER (ORDER BY o.created_ts, o.asset_id)::VARCHAR, 5, '0') AS wo_id,
       o.asset_id, o.site_id, o.created_ts, o.closed_ts, o.wo_type, o.fault_code, o.description, o.status,
       o.spare_part_kit, o.cost_eur,
       t.names[o.tech_idx]::VARCHAR AS technician
FROM orders o
CROSS JOIN technicians t;

-- Back to normal size after data generation
ALTER WAREHOUSE EPOWER_COMPUTE SET WAREHOUSE_SIZE = 'SMALL';

SELECT wo_id, asset_id, created_ts, wo_type, fault_code, status, description
FROM EPOWER_DEMO.EIS_GOLD.EIS_WORK_ORDERS WHERE site_id = 'S001' AND wo_type = 'CORRECTIVE' ORDER BY created_ts;

## 5. Documents: Contracts & OEM Manuals → AI-Ready

Much of EIS's critical knowledge sits in documents: **contract guarantees and penalties**, **OEM maintenance procedures** and **program rules**. Three steps make it usable:

1. **Parse** all documents with `AI_PARSE_DOCUMENT` → `EIS_DOCS_PARSED`
2. **Extract** the guarantee clauses from the contracts with `AI_EXTRACT` → `EIS_CONTRACT_CLAUSES`, and compare them with the contract table. *Unstructured contract text becomes structured, queryable data in one SQL statement.*
3. **Index** everything in Cortex Search (`SEARCH_EIS_DOCS`) so the agent can retrieve procedures and clauses (RAG)

| Document | Content |
|----------|---------|
| `contract_klinikum_hamburg_nord.md` | 97 % availability per CHP unit, €45k penalty per point |
| `contract_chemiepark_leuna.md` | 96 % average availability, COP ≥ 3.0, EIS Flex 60/40 |
| `contract_waermenetz_altona.md` | 99.5 % supply availability, EIS Flex 70/30 |
| `om_manual_chp_ecopower_ec400.md` | Fault codes incl. **F-217**, cleaning procedure, spare part kit |
| `om_manual_heatpump_thermaflux_hp2000.md` | Heat pump fault codes incl. H-512 |
| `om_manual_thermal_storage.md` | Storage limits, supply security reserve |
| `eis_flex_program_guide.md` | EIS Flex rules, economics, eligibility |

The documents are in this module's `docs/` folder, not in `demo_data/`, so they don't end up in the EPOWER agents' search services.

In [ ]:
# Upload the EIS documents from this module's docs/ folder to the encrypted stage
import os

stage_name = '@EPOWER_DEMO.EIS_GOLD.EIS_DOCS_STAGE'
for file in sorted(os.listdir('docs')):
    if file.endswith(('.md', '.pdf')):
        session.file.put(os.path.join('docs', file), f'{stage_name}/', auto_compress=False, overwrite=True)
        print(f'  uploaded docs/{file}')

session.sql('ALTER STAGE EPOWER_DEMO.EIS_GOLD.EIS_DOCS_STAGE REFRESH').collect()
print('\nDocuments uploaded to stage')

In [ ]:
%%sql
-- Step 1: parse all documents with AI_PARSE_DOCUMENT (takes ~1 minute)
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_DOCS_PARSED
    COMMENT = 'Parsed EIS documents: contracts, OEM manuals, EIS Flex program guide'
AS
SELECT relative_path,
       CASE WHEN relative_path ILIKE 'contract_%' THEN 'CONTRACT'
            WHEN relative_path ILIKE 'om_manual_%' THEN 'OEM_MANUAL'
            ELSE 'PROGRAM_GUIDE' END AS doc_type,
       BUILD_STAGE_FILE_URL('@EPOWER_DEMO.EIS_GOLD.EIS_DOCS_STAGE', relative_path) AS file_url,
       AI_PARSE_DOCUMENT(TO_FILE('@EPOWER_DEMO.EIS_GOLD.EIS_DOCS_STAGE', relative_path), {'mode': 'LAYOUT'}):content::STRING AS content
FROM DIRECTORY(@EPOWER_DEMO.EIS_GOLD.EIS_DOCS_STAGE);

SELECT relative_path, doc_type, LENGTH(content) AS characters FROM EPOWER_DEMO.EIS_GOLD.EIS_DOCS_PARSED ORDER BY doc_type, relative_path;

In [ ]:
%%sql
-- Step 2: extract guarantee clauses from the contract documents with AI_EXTRACT
CREATE OR REPLACE TABLE EPOWER_DEMO.EIS_GOLD.EIS_CONTRACT_CLAUSES
    COMMENT = 'Guarantee clauses extracted from contract documents with AI_EXTRACT'
AS
WITH extracted AS (
    SELECT relative_path,
           AI_EXTRACT(
               text => content,
               responseFormat => {
                   'contract_id': 'What is the contract number?',
                   'site_id': 'What is the site ID (format S followed by three digits)?',
                   'annual_fee_eur': 'What is the annual base, service or network operation fee in EUR? Answer with the number only.',
                   'availability_guarantee_pct': 'What minimum availability percentage is guaranteed? Answer with the number only.',
                   'availability_measurement': 'Is the availability guarantee measured per unit, as an average across units, or at heat supply / network level?',
                   'availability_penalty_eur': 'What is the penalty in EUR per percentage point of missed availability? Answer with the number only.',
                   'flex_participation': 'Does the site participate in the EIS Flex program? Answer yes or no.',
                   'flex_share_eis_pct': 'What percentage of the flexibility margin goes to EPOWER Infrastructure Solutions (the Contractor, Provider or Operator)? Answer with the number only, or none.'
               }
           ) AS extraction
    FROM EPOWER_DEMO.EIS_GOLD.EIS_DOCS_PARSED
    WHERE doc_type = 'CONTRACT'
)
SELECT relative_path,
       extraction:response:contract_id::STRING AS contract_id,
       extraction:response:site_id::STRING AS site_id,
       TRY_TO_DECIMAL(REGEXP_REPLACE(extraction:response:annual_fee_eur::STRING, '[^0-9.]', ''), 14, 2) AS annual_fee_eur,
       TRY_TO_DECIMAL(REGEXP_REPLACE(extraction:response:availability_guarantee_pct::STRING, '[^0-9.]', ''), 6, 2) AS guaranteed_availability_pct,
       extraction:response:availability_measurement::STRING AS availability_measurement,
       TRY_TO_DECIMAL(REGEXP_REPLACE(extraction:response:availability_penalty_eur::STRING, '[^0-9.]', ''), 14, 2) AS penalty_availability_eur_per_pt,
       extraction:response:flex_participation::STRING ILIKE 'yes%' AS flex_participation,
       TRY_TO_DECIMAL(REGEXP_REPLACE(extraction:response:flex_share_eis_pct::STRING, '[^0-9.]', ''), 6, 2) AS flex_share_eis_pct,
       extraction AS raw_extraction
FROM extracted;

SELECT contract_id, site_id, annual_fee_eur, guaranteed_availability_pct, availability_measurement,
       penalty_availability_eur_per_pt, flex_participation, flex_share_eis_pct
FROM EPOWER_DEMO.EIS_GOLD.EIS_CONTRACT_CLAUSES ORDER BY site_id;

In [ ]:
%%sql
-- Check: do the terms extracted from the documents match the contract table?
SELECT c.contract_id,
       c.site_id,
       c.guaranteed_availability_pct      AS table_availability_pct,
       x.guaranteed_availability_pct      AS document_availability_pct,
       c.penalty_availability_eur_per_pt  AS table_penalty_eur,
       x.penalty_availability_eur_per_pt  AS document_penalty_eur,
       x.availability_measurement         AS document_measurement,
       IFF(c.guaranteed_availability_pct = x.guaranteed_availability_pct
           AND c.penalty_availability_eur_per_pt = x.penalty_availability_eur_per_pt, 'MATCH', 'CHECK') AS result
FROM EPOWER_DEMO.EIS_GOLD.EIS_CONTRACTS c
JOIN EPOWER_DEMO.EIS_GOLD.EIS_CONTRACT_CLAUSES x ON x.site_id = c.site_id
ORDER BY c.site_id;

In [ ]:
%%sql
-- Step 3: Cortex Search over all EIS documents (RAG for the agent)
CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EIS_GOLD.SEARCH_EIS_DOCS
    ON content
    ATTRIBUTES relative_path, doc_type
    WAREHOUSE = EPOWER_COMPUTE
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT relative_path, doc_type, file_url, content FROM EPOWER_DEMO.EIS_GOLD.EIS_DOCS_PARSED
);

-- Try it: what does the OEM manual say about F-217?
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'EPOWER_DEMO.EIS_GOLD.SEARCH_EIS_DOCS',
        '{"query": "F-217 exhaust heat exchanger fouling cleaning procedure", "columns": ["relative_path"], "limit": 2}'
    )
):results AS results;

## 6. Business Views: From Telemetry to Decisions

Plain SQL views turn raw data into the KPIs EIS management cares about. There are no pipelines to maintain: the views always reflect the current tables.

| View | Grain | Answers |
|------|-------|---------|
| `V_EIS_ASSET_DAILY` | asset × day | Running/fault hours, output, efficiency per day |
| `V_EIS_ASSET_HEALTH` | asset (last 30 days) | Availability, efficiency trend, fault events, open work orders, **health score** |
| `V_EIS_GUARANTEE_STATUS` | contract (last 30 days) | Actual vs. guaranteed availability and efficiency, **€ penalty exposure**, status OK / WATCH / AT_RISK |
| `V_EIS_FLEX_MARGIN` | site × day | Heat pump electricity cost normal vs. EIS Flex: **realised** margin (Flex sites), **missed** margin (baseline sites), EIS/customer split |
| `V_EIS_DISPATCH_PLAN` | Flex site × hour of the plan day | **Next-day heat pump schedule** from the latest day-ahead prices |

**Guarantee logic:** contracts measure guarantees per quarter. The view uses the **rolling last 30 days** as the leading indicator: *"if the current performance continues, this is the penalty for the quarter"*. Penalties count every full or partial missed percentage point (`CEIL`), as defined in the contracts.

In [ ]:
%%sql
-- Daily asset KPIs (compact grain for the semantic view)
CREATE OR REPLACE VIEW EPOWER_DEMO.EIS_GOLD.V_EIS_ASSET_DAILY
    COMMENT = 'Daily KPIs per asset: running and fault hours, heat/electricity output, gas use, efficiency, alarms'
AS
SELECT day, asset_id, site_id, asset_type,
       COUNT_IF(status IN ('RUNNING', 'ACTIVE')) AS running_hours,
       COUNT_IF(status = 'FAULT') AS fault_hours,
       ROUND(SUM(heat_output_mwh), 2) AS heat_output_mwh,
       ROUND(SUM(el_output_mwh), 2) AS el_output_mwh,
       ROUND(SUM(el_consumption_mwh), 2) AS el_consumption_mwh,
       ROUND(SUM(gas_consumption_mwh), 2) AS gas_consumption_mwh,
       ROUND(AVG(efficiency_pct), 2) AS avg_efficiency_pct,
       ROUND(AVG(storage_soc_pct), 1) AS avg_storage_soc_pct,
       COUNT_IF(alarm_level = 'WARNING') AS warning_count,
       COUNT_IF(alarm_level = 'TRIP') AS trip_hours
FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSET_TELEMETRY
GROUP BY day, asset_id, site_id, asset_type;

-- Asset health over the last 30 days
CREATE OR REPLACE VIEW EPOWER_DEMO.EIS_GOLD.V_EIS_ASSET_HEALTH
    COMMENT = 'Asset health (rolling 30 days): availability, efficiency trend, faults, open work orders, health score'
AS
WITH window_start AS (
    SELECT DATEADD(DAY, -30, CURRENT_DATE()) AS d0
),
daily AS (
    SELECT ad.*, DATEDIFF(DAY, w.d0, ad.day) AS day_index
    FROM EPOWER_DEMO.EIS_GOLD.V_EIS_ASSET_DAILY ad, window_start w
    WHERE ad.day >= w.d0 AND ad.asset_type <> 'THERMAL_STORAGE'
),
agg AS (
    SELECT asset_id,
           COUNT(*) * 24 AS hours,
           SUM(fault_hours) AS fault_hours,
           ROUND(AVG(avg_efficiency_pct), 2) AS avg_efficiency_30d,
           ROUND(AVG(IFF(day_index >= 23, avg_efficiency_pct, NULL)), 2) AS avg_efficiency_last_7d,
           REGR_SLOPE(avg_efficiency_pct, day_index) AS efficiency_slope,
           SUM(warning_count) AS warnings_30d
    FROM daily
    GROUP BY asset_id
),
events AS (
    SELECT f.asset_id, COUNT(*) AS fault_events_30d, MODE(f.fault_code) AS top_fault_code
    FROM EPOWER_DEMO.EIS_GOLD.EIS_FAULT_EVENTS f, window_start w
    WHERE f.start_ts >= w.d0
    GROUP BY f.asset_id
),
wo AS (
    SELECT asset_id, COUNT_IF(status = 'OPEN') AS open_work_orders,
           MAX_BY(IFF(status = 'OPEN', description, NULL), IFF(status = 'OPEN', created_ts, NULL)) AS open_work_order_note
    FROM EPOWER_DEMO.EIS_GOLD.EIS_WORK_ORDERS GROUP BY asset_id
),
scored AS (
    SELECT a.asset_id, a.site_id, a.asset_code, a.asset_type, a.model, a.nominal_efficiency_pct,
           ROUND(100 * (1 - g.fault_hours / g.hours), 2) AS availability_30d_pct,
           g.fault_hours AS fault_hours_30d,
           g.avg_efficiency_30d,
           g.avg_efficiency_last_7d,
           ROUND(g.efficiency_slope, 3) AS efficiency_trend_pp_per_day,
           COALESCE(e.fault_events_30d, 0) AS fault_events_30d,
           e.top_fault_code,
           g.warnings_30d,
           COALESCE(w.open_work_orders, 0) AS open_work_orders,
           w.open_work_order_note,
           -- slope normalised to CHP scale (heat pumps report COP x 100)
           COALESCE(g.efficiency_slope, 0) * 88 / a.nominal_efficiency_pct AS norm_slope
    FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSETS a
    JOIN agg g ON g.asset_id = a.asset_id
    LEFT JOIN events e ON e.asset_id = a.asset_id
    LEFT JOIN wo w ON w.asset_id = a.asset_id
)
SELECT s.* EXCLUDE (norm_slope),
       -- Days until a declining CHP falls below the typical 85 % efficiency guarantee
       CASE WHEN s.asset_type = 'CHP' AND s.efficiency_trend_pp_per_day < -0.02
            THEN GREATEST(0, ROUND((s.avg_efficiency_last_7d - 85) / -s.efficiency_trend_pp_per_day)) END AS days_until_below_85_pct,
       ROUND(GREATEST(0, LEAST(100,
             100 - (100 - s.availability_30d_pct) * 5
                 - GREATEST(0, -s.norm_slope) * 100
                 - s.fault_events_30d * 3
                 - s.warnings_30d * 0.5))) AS health_score,
       CASE WHEN health_score < 60 THEN 'CRITICAL' WHEN health_score < 80 THEN 'WARNING' ELSE 'GOOD' END AS health_status
FROM scored s;

SELECT asset_id, availability_30d_pct, avg_efficiency_30d, efficiency_trend_pp_per_day, fault_events_30d,
       top_fault_code, open_work_orders, days_until_below_85_pct, health_score, health_status
FROM EPOWER_DEMO.EIS_GOLD.V_EIS_ASSET_HEALTH ORDER BY health_score LIMIT 10;

In [ ]:
%%sql
-- Guarantee status per contract: actual (rolling 30 days) vs. guaranteed, and EUR penalty exposure
CREATE OR REPLACE VIEW EPOWER_DEMO.EIS_GOLD.V_EIS_GUARANTEE_STATUS
    COMMENT = 'Contract guarantee status (rolling 30 days as quarterly leading indicator): availability/efficiency gap and EUR penalty exposure'
AS
WITH units AS (
    SELECT c.contract_id, h.asset_id, h.asset_type, h.availability_30d_pct, h.avg_efficiency_30d,
           GREATEST(0, c.guaranteed_availability_pct - h.availability_30d_pct) AS unit_gap
    FROM EPOWER_DEMO.EIS_GOLD.EIS_CONTRACTS c
    JOIN EPOWER_DEMO.EIS_GOLD.V_EIS_ASSET_HEALTH h
      ON h.site_id = c.site_id
     AND (h.asset_type = 'CHP' OR (c.availability_scope = 'CHP_HP' AND h.asset_type = 'HEAT_PUMP'))
),
avail AS (
    SELECT contract_id,
           MIN(availability_30d_pct) AS min_unit_availability_pct,
           ROUND(AVG(availability_30d_pct), 2) AS avg_unit_availability_pct,
           MIN_BY(asset_id, availability_30d_pct) AS worst_unit,
           SUM(CEIL(unit_gap)) AS sum_unit_gap_points
    FROM units GROUP BY contract_id
),
eff AS (
    SELECT c.contract_id, ROUND(AVG(h.avg_efficiency_30d), 2) AS actual_efficiency_pct
    FROM EPOWER_DEMO.EIS_GOLD.EIS_CONTRACTS c
    JOIN EPOWER_DEMO.EIS_GOLD.V_EIS_ASSET_HEALTH h
      ON h.site_id = c.site_id AND h.asset_type = c.efficiency_scope
    GROUP BY c.contract_id
),
calc AS (
    SELECT c.contract_id, c.site_id, s.site_name, s.customer_name, c.contract_type,
           c.guaranteed_availability_pct, c.availability_basis,
           CASE c.availability_basis
               WHEN 'PER_UNIT' THEN a.min_unit_availability_pct
               WHEN 'AVERAGE'  THEN a.avg_unit_availability_pct
               ELSE 100 END AS actual_availability_pct,
           a.worst_unit,
           c.guaranteed_efficiency_pct, c.efficiency_scope,
           e.actual_efficiency_pct,
           c.penalty_availability_eur_per_pt, c.penalty_efficiency_eur_per_pt,
           CASE c.availability_basis
               WHEN 'PER_UNIT' THEN a.sum_unit_gap_points
               WHEN 'AVERAGE'  THEN CEIL(GREATEST(0, c.guaranteed_availability_pct - a.avg_unit_availability_pct))
               ELSE 0 END AS availability_gap_points,
           CEIL(GREATEST(0, c.guaranteed_efficiency_pct - COALESCE(e.actual_efficiency_pct, c.guaranteed_efficiency_pct))) AS efficiency_gap_points,
           c.annual_fee_eur, c.terms_source
    FROM EPOWER_DEMO.EIS_GOLD.EIS_CONTRACTS c
    JOIN EPOWER_DEMO.EIS_GOLD.EIS_SITES s ON s.site_id = c.site_id
    LEFT JOIN avail a ON a.contract_id = c.contract_id
    LEFT JOIN eff e ON e.contract_id = c.contract_id
)
SELECT calc.*,
       ROUND(actual_availability_pct - guaranteed_availability_pct, 2) AS availability_headroom_pts,
       ROUND(actual_efficiency_pct - guaranteed_efficiency_pct, 2) AS efficiency_headroom_pts,
       availability_gap_points * penalty_availability_eur_per_pt AS availability_penalty_exposure_eur,
       efficiency_gap_points * penalty_efficiency_eur_per_pt AS efficiency_penalty_exposure_eur,
       availability_penalty_exposure_eur + efficiency_penalty_exposure_eur AS total_penalty_exposure_eur,
       CASE WHEN availability_headroom_pts < 0 OR efficiency_headroom_pts < 0 THEN 'AT_RISK'
            WHEN (availability_basis <> 'SUPPLY' AND availability_headroom_pts < 1.0)
              OR COALESCE(efficiency_headroom_pts, 99) < 1.0 THEN 'WATCH'
            ELSE 'OK' END AS guarantee_status,
       'Rolling 30 days' AS measurement_window
FROM calc;

SELECT contract_id, site_name, guarantee_status, guaranteed_availability_pct, actual_availability_pct, worst_unit,
       guaranteed_efficiency_pct, actual_efficiency_pct, total_penalty_exposure_eur
FROM EPOWER_DEMO.EIS_GOLD.V_EIS_GUARANTEE_STATUS
WHERE guarantee_status <> 'OK' ORDER BY total_penalty_exposure_eur DESC;

In [ ]:
%%sql
-- EIS Flex margin per site and day: realised on Flex sites, missed on baseline sites
CREATE OR REPLACE VIEW EPOWER_DEMO.EIS_GOLD.V_EIS_FLEX_MARGIN
    COMMENT = 'Daily EIS Flex margin per site: heat pump electricity cost normal vs. EIS Flex; realised (Flex sites) and missed (baseline sites); EIS/customer split'
AS
SELECT d.day,
       d.site_id,
       s.site_name,
       s.site_type,
       s.city,
       d.flex_enabled,
       IFF(d.flex_enabled, 'EIS_FLEX', 'BASELINE') AS program_status,
       ROUND(SUM(d.heat_demand_mwh), 2) AS heat_demand_mwh,
       ROUND(SUM(d.hp_heat_normal_mwh), 2) AS hp_heat_mwh,
       -- Average power price the heat pump paid: normal vs. EIS Flex (same heat, different hours)
       ROUND(SUM(d.hp_heat_normal_mwh * d.price_eur_mwh) / NULLIF(SUM(d.hp_heat_normal_mwh), 0), 2) AS avg_price_paid_normal_eur_mwh,
       ROUND(SUM(d.hp_heat_flex_mwh * d.price_eur_mwh) / NULLIF(SUM(d.hp_heat_flex_mwh), 0), 2) AS avg_price_paid_flex_eur_mwh,
       ROUND(AVG(d.price_eur_mwh), 2) AS avg_day_price_eur_mwh,
       COUNT_IF(d.hp_heat_flex_mwh > 0.001) AS flex_hp_running_hours,
       ROUND(SUM(d.hp_cost_normal_eur), 2) AS hp_cost_normal_eur,
       ROUND(SUM(d.hp_cost_flex_eur), 2) AS hp_cost_flex_eur,
       ROUND(SUM(d.flex_margin_eur), 2) AS flex_margin_eur,
       ROUND(SUM(IFF(d.flex_enabled, d.flex_margin_eur, 0)), 2) AS realised_flex_margin_eur,
       ROUND(SUM(IFF(d.flex_enabled, 0, d.flex_margin_eur)), 2) AS missed_flex_margin_eur,
       -- Margin split: contract share for Flex sites, 70 % assumed for baseline sites
       COALESCE(c.flex_share_eis_pct, 70) AS eis_share_pct,
       ROUND(SUM(d.flex_margin_eur) * COALESCE(c.flex_share_eis_pct, 70) / 100, 2) AS eis_margin_eur,
       ROUND(SUM(d.flex_margin_eur) * (100 - COALESCE(c.flex_share_eis_pct, 70)) / 100, 2) AS customer_margin_eur,
       ROUND(SUM(GREATEST(d.flex_storage_flow_mwh, 0)), 2) AS heat_shifted_via_storage_mwh
FROM EPOWER_DEMO.EIS_GOLD.EIS_SITE_DISPATCH_HOURLY d
JOIN EPOWER_DEMO.EIS_GOLD.EIS_SITES s ON s.site_id = d.site_id
LEFT JOIN EPOWER_DEMO.EIS_GOLD.EIS_CONTRACTS c ON c.site_id = d.site_id
GROUP BY d.day, d.site_id, s.site_name, s.site_type, s.city, d.flex_enabled, c.flex_share_eis_pct;

-- Realised vs. missed margin, last 30 days (annualised for the business case)
SELECT program_status, COUNT(DISTINCT site_id) AS sites,
       ROUND(SUM(flex_margin_eur)) AS margin_eur_30d,
       ROUND(SUM(flex_margin_eur) * 365 / 30) AS annualised_eur,
       ROUND(SUM(hp_heat_mwh * avg_price_paid_normal_eur_mwh) / NULLIF(SUM(hp_heat_mwh), 0), 1) AS avg_price_paid_normal,
       ROUND(SUM(hp_heat_mwh * avg_price_paid_flex_eur_mwh) / NULLIF(SUM(hp_heat_mwh), 0), 1) AS avg_price_paid_flex
FROM EPOWER_DEMO.EIS_GOLD.V_EIS_FLEX_MARGIN
WHERE day >= DATEADD(DAY, -30, CURRENT_DATE())
GROUP BY program_status;

In [ ]:
%%sql
-- Next-day heat pump schedule for EIS Flex sites, based on the latest available day-ahead prices.
-- The daily Module 1 task loads tomorrow's prices at ~17:00 CET; before that, the plan uses the latest available day.
-- Same rule as in section 4b: produce the expected heat pump heat in the cheapest hours, storage bridges the rest.
CREATE OR REPLACE VIEW EPOWER_DEMO.EIS_GOLD.V_EIS_DISPATCH_PLAN
    COMMENT = 'Hourly EIS Flex heat pump schedule per Flex site for the plan day (tomorrow if prices are published, else latest available day)'
AS
WITH params AS (
    SELECT hp_cop FROM EPOWER_DEMO.EIS_GOLD.EIS_PARAMETERS
),
prices AS (
    SELECT DATE_TRUNC('HOUR', CONVERT_TIMEZONE('UTC', 'Europe/Berlin', start_time)) AS ts,
           AVG(price_eur_mwh) AS price_eur_mwh
    FROM EPOWER_DEMO.EPOWER_GOLD.MART_DAY_AHEAD_PRICES
    GROUP BY 1
),
plan_day AS (
    SELECT COALESCE(MIN(IFF(ts::DATE > CURRENT_DATE(), ts::DATE, NULL)), MAX(ts::DATE)) AS plan_date
    FROM prices
),
hp_capacity AS (
    SELECT site_id, SUM(th_capacity_mw) AS hp_cap
    FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSETS WHERE asset_type = 'HEAT_PUMP' GROUP BY site_id
),
expected AS (
    -- Forecast = average of the same hour over the last 7 days
    SELECT site_id, HOUR(ts) AS hour_of_day,
           AVG(heat_demand_mwh) AS heat_demand_mwh,
           AVG(hp_heat_normal_mwh) AS hp_heat_normal_mwh
    FROM EPOWER_DEMO.EIS_GOLD.EIS_SITE_DISPATCH_HOURLY
    WHERE ts >= DATEADD(DAY, -7, CURRENT_DATE())
    GROUP BY 1, 2
),
plan AS (
    SELECT pd.plan_date, p.ts, s.site_id, s.site_name, s.storage_capacity_mwh, c.hp_cap,
           p.price_eur_mwh, e.heat_demand_mwh, e.hp_heat_normal_mwh,
           RANK() OVER (PARTITION BY s.site_id ORDER BY p.price_eur_mwh) AS price_rank_in_day,
           SUM(e.hp_heat_normal_mwh) OVER (PARTITION BY s.site_id) AS hp_heat_day,
           COALESCE(SUM(c.hp_cap) OVER (PARTITION BY s.site_id ORDER BY p.price_eur_mwh, p.ts
                                        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) AS hp_cap_cheaper_hours
    FROM prices p
    JOIN plan_day pd ON p.ts::DATE = pd.plan_date
    JOIN EPOWER_DEMO.EIS_GOLD.EIS_SITES s ON s.flex_enabled
    JOIN hp_capacity c ON c.site_id = s.site_id
    JOIN expected e ON e.site_id = s.site_id AND e.hour_of_day = HOUR(p.ts)
),
schedule AS (
    SELECT pl.*,
           IFF(pl.hp_heat_day > 0, LEAST(1, pl.storage_capacity_mwh * 0.85 / pl.hp_heat_day), 0) AS shift_share,
           LEAST(pl.hp_cap, GREATEST(0, pl.hp_heat_day - pl.hp_cap_cheaper_hours)) AS hp_heat_cheapest
    FROM plan pl
),
planned AS (
    SELECT sc.*,
           (1 - sc.shift_share) * sc.hp_heat_normal_mwh + sc.shift_share * sc.hp_heat_cheapest AS planned_hp_heat
    FROM schedule sc
),
with_storage AS (
    SELECT pd.*,
           pd.planned_hp_heat - pd.hp_heat_normal_mwh AS storage_flow,
           SUM(pd.planned_hp_heat - pd.hp_heat_normal_mwh) OVER (PARTITION BY pd.site_id ORDER BY pd.ts) AS storage_level
    FROM planned pd
)
SELECT w.plan_date,
       w.plan_date > CURRENT_DATE() AS is_next_day,
       w.ts AS plan_hour,
       HOUR(w.ts) AS hour_of_day,
       w.site_id,
       w.site_name,
       ROUND(w.price_eur_mwh, 2) AS price_eur_mwh,
       w.price_rank_in_day,
       w.planned_hp_heat > 0.01 AS heat_pump_on,
       ROUND(w.heat_demand_mwh, 2) AS expected_heat_demand_mwh,
       ROUND(w.hp_heat_normal_mwh, 2) AS normal_hp_heat_mwh,
       ROUND(w.planned_hp_heat, 2) AS planned_hp_heat_mwh,
       CASE WHEN w.storage_flow > 0.01 THEN 'CHARGE' WHEN w.storage_flow < -0.01 THEN 'DISCHARGE' ELSE 'HOLD' END AS storage_action,
       ROUND(w.storage_flow, 2) AS storage_flow_mwh,
       ROUND(10 + 100 * (w.storage_level - LEAST(0, MIN(w.storage_level) OVER (PARTITION BY w.site_id)))
                  / NULLIF(w.storage_capacity_mwh, 0), 1) AS expected_storage_soc_pct,
       ROUND(w.hp_heat_normal_mwh / p.hp_cop * w.price_eur_mwh, 2) AS normal_hp_cost_eur,
       ROUND(w.planned_hp_heat / p.hp_cop * w.price_eur_mwh, 2) AS planned_hp_cost_eur,
       ROUND(normal_hp_cost_eur - planned_hp_cost_eur, 2) AS expected_margin_eur
FROM with_storage w
CROSS JOIN params p;

-- Example: the schedule for Waermenetz Hamburg-Altona
SELECT plan_date, is_next_day, hour_of_day, price_eur_mwh, heat_pump_on, planned_hp_heat_mwh,
       storage_action, expected_storage_soc_pct, expected_margin_eur
FROM EPOWER_DEMO.EIS_GOLD.V_EIS_DISPATCH_PLAN WHERE site_id = 'S003' ORDER BY plan_hour;

## 7. Semantic View: The Business Vocabulary of EIS

The semantic view `EIS_OPERATIONS_SEMANTIC_VIEW` tells Cortex Analyst how the data relates and what the business terms mean. With it, the agent can translate *"Which Klinikum BHKW is at risk?"* into correct SQL.

It deliberately exposes only the **six business-level objects** the questions need. Raw telemetry and work orders stay behind the views.

- **Relationships:** everything hangs off the site; asset health hangs off the asset.
- **Synonyms:** German and English terms (*BHKW* = CHP, *Wärmepumpe* = heat pump, *Pönale* = penalty, *Verfügbarkeit* = availability).
- **AI_SQL_GENERATION instructions:** domain rules, e.g. that heat pump efficiency is COP × 100, that EIS Flex only shifts heat pump operation, or that "missed margin" means baseline sites.

```mermaid
flowchart LR
    sites --> assets
    assets --> asset_health
    sites --> guarantees
    sites --> flex_margin
    sites --> heat_pump_schedule
```

In [ ]:
%%sql
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EIS_GOLD.EIS_OPERATIONS_SEMANTIC_VIEW
  TABLES (
    sites AS EPOWER_DEMO.EIS_GOLD.EIS_SITES
      PRIMARY KEY (site_id)
      WITH SYNONYMS ('Standort', 'customer site', 'energy centre')
      COMMENT = 'EIS customer sites: hospitals, industrial parks, housing estates, district heating networks, campuses',
    assets AS EPOWER_DEMO.EIS_GOLD.EIS_ASSETS
      PRIMARY KEY (asset_id)
      WITH SYNONYMS ('Anlage', 'unit', 'plant')
      COMMENT = 'Asset register: CHP units, heat pumps, boilers, thermal storage',
    asset_health AS EPOWER_DEMO.EIS_GOLD.V_EIS_ASSET_HEALTH
      PRIMARY KEY (asset_id)
      COMMENT = 'Asset health over the last 30 days: availability, efficiency trend, faults, open work order, health score',
    guarantees AS EPOWER_DEMO.EIS_GOLD.V_EIS_GUARANTEE_STATUS
      PRIMARY KEY (contract_id)
      WITH SYNONYMS ('contract', 'Vertrag', 'guarantee status', 'contract risk')
      COMMENT = 'Contract guarantees vs. actual performance over the rolling last 30 days and EUR penalty exposure',
    flex_margin AS EPOWER_DEMO.EIS_GOLD.V_EIS_FLEX_MARGIN
      PRIMARY KEY (site_id, day)
      WITH SYNONYMS ('EIS Flex margin', 'flex saving')
      COMMENT = 'Daily EIS Flex margin per site: heat pump electricity cost normal vs. EIS Flex',
    heat_pump_schedule AS EPOWER_DEMO.EIS_GOLD.V_EIS_DISPATCH_PLAN
      PRIMARY KEY (site_id, plan_hour)
      WITH SYNONYMS ('Fahrplan', 'dispatch plan', 'schedule')
      COMMENT = 'Hourly EIS Flex heat pump schedule for the plan day (next day when prices are published)'
  )
  RELATIONSHIPS (
    assets_to_sites AS assets (site_id) REFERENCES sites,
    health_to_assets AS asset_health (asset_id) REFERENCES assets,
    guarantees_to_sites AS guarantees (site_id) REFERENCES sites,
    flex_to_sites AS flex_margin (site_id) REFERENCES sites,
    schedule_to_sites AS heat_pump_schedule (site_id) REFERENCES sites
  )
  FACTS (
    asset_health.availability_30d AS availability_30d_pct
      WITH SYNONYMS ('Verfuegbarkeit', 'availability') COMMENT = 'Technical availability over the last 30 days in percent',
    asset_health.efficiency_30d AS avg_efficiency_30d COMMENT = 'Average efficiency over the last 30 days (heat pumps: COP x 100)',
    asset_health.efficiency_trend AS efficiency_trend_pp_per_day COMMENT = 'Efficiency trend in percentage points per day (negative = degrading)',
    asset_health.fault_events AS fault_events_30d COMMENT = 'Number of fault trips in the last 30 days',
    asset_health.days_until_below_85 AS days_until_below_85_pct COMMENT = 'Projected days until a degrading CHP falls below 85 percent efficiency',
    asset_health.health_score AS health_score COMMENT = 'Health score 0-100 (lower is worse)',
    guarantees.guaranteed_availability AS guaranteed_availability_pct COMMENT = 'Contractually guaranteed availability in percent',
    guarantees.actual_availability AS actual_availability_pct COMMENT = 'Actual availability (rolling 30 days) according to the contract basis',
    guarantees.guaranteed_efficiency AS guaranteed_efficiency_pct COMMENT = 'Contractually guaranteed efficiency',
    guarantees.actual_efficiency AS actual_efficiency_pct COMMENT = 'Actual efficiency (rolling 30 days)',
    guarantees.penalty_exposure AS total_penalty_exposure_eur
      WITH SYNONYMS ('Poenale', 'penalty', 'Vertragsstrafe') COMMENT = 'Projected quarterly penalty in EUR if the current performance continues',
    flex_margin.price_paid_normal AS avg_price_paid_normal_eur_mwh COMMENT = 'Average power price the heat pump paid in normal operation in EUR/MWh',
    flex_margin.price_paid_flex AS avg_price_paid_flex_eur_mwh COMMENT = 'Average power price the heat pump paid with EIS Flex in EUR/MWh',
    flex_margin.margin AS flex_margin_eur COMMENT = 'Saved heat pump electricity cost in EUR (realised on Flex sites, missed on baseline sites)',
    flex_margin.eis_margin AS eis_margin_eur COMMENT = 'EIS share of the flex margin in EUR',
    heat_pump_schedule.price AS price_eur_mwh COMMENT = 'Day-ahead price of the plan hour in EUR/MWh',
    heat_pump_schedule.planned_heat AS planned_hp_heat_mwh COMMENT = 'Planned heat pump heat output in MWh',
    heat_pump_schedule.storage_soc AS expected_storage_soc_pct COMMENT = 'Expected storage state of charge in percent',
    heat_pump_schedule.expected_saving AS expected_margin_eur COMMENT = 'Expected saving vs. normal operation in EUR'
  )
  DIMENSIONS (
    sites.site_id AS site_id COMMENT = 'Site ID, e.g. S001',
    sites.site_name AS site_name WITH SYNONYMS ('Standortname', 'site') COMMENT = 'Site name, e.g. Klinikum Hamburg-Nord, Waermenetz Hamburg-Altona',
    sites.site_type AS site_type COMMENT = 'HOSPITAL, INDUSTRIAL_PARK, HOUSING, DISTRICT_HEATING, CAMPUS',
    sites.city AS city WITH SYNONYMS ('Stadt') COMMENT = 'City',
    sites.flex_enabled AS flex_enabled WITH SYNONYMS ('EIS Flex enrolled') COMMENT = 'TRUE if the site is enrolled in EIS Flex',
    assets.asset_id AS asset_id COMMENT = 'Asset ID = site ID + unit code, e.g. S001-CHP-2',
    assets.asset_code AS asset_code COMMENT = 'Unit code within the site, e.g. CHP-2, HP-1',
    assets.asset_type AS asset_type
      WITH SYNONYMS ('BHKW', 'Waermepumpe', 'Kessel', 'Waermespeicher')
      COMMENT = 'CHP (BHKW), HEAT_PUMP (Waermepumpe), BOILER (Kessel), THERMAL_STORAGE (Waermespeicher)',
    assets.model AS model COMMENT = 'Model, e.g. EcoPower EC400',
    asset_health.health_status AS health_status COMMENT = 'GOOD, WARNING or CRITICAL',
    asset_health.top_fault_code AS top_fault_code COMMENT = 'Most frequent fault code in the last 30 days, e.g. F-217',
    asset_health.open_work_order_note AS open_work_order_note COMMENT = 'Technician note of the open work order, if any',
    guarantees.contract_id AS contract_id COMMENT = 'Contract number, e.g. EIS-C-0001',
    guarantees.guarantee_status AS guarantee_status COMMENT = 'OK, WATCH (less than 1 point headroom) or AT_RISK (below guarantee)',
    guarantees.availability_basis AS availability_basis COMMENT = 'PER_UNIT, AVERAGE or SUPPLY',
    guarantees.worst_unit AS worst_unit COMMENT = 'Unit with the lowest availability under the contract',
    flex_margin.margin_day AS day COMMENT = 'Day of the flex margin',
    flex_margin.program_status AS program_status COMMENT = 'EIS_FLEX (enrolled) or BASELINE (not enrolled)',
    heat_pump_schedule.plan_date AS plan_date COMMENT = 'Date the schedule is for',
    heat_pump_schedule.is_next_day AS is_next_day COMMENT = 'TRUE if the schedule is for tomorrow, FALSE if tomorrow''s prices are not published yet',
    heat_pump_schedule.hour_of_day AS hour_of_day COMMENT = 'Hour of day 0-23 (CET)',
    heat_pump_schedule.heat_pump_on AS heat_pump_on COMMENT = 'TRUE if the heat pump is scheduled to run in this hour',
    heat_pump_schedule.storage_action AS storage_action COMMENT = 'CHARGE, DISCHARGE or HOLD'
  )
  METRICS (
    guarantees.total_penalty_exposure AS SUM(guarantees.penalty_exposure)
      WITH SYNONYMS ('total penalty', 'Poenale gesamt') COMMENT = 'Total projected penalty exposure in EUR',
    flex_margin.total_flex_margin AS SUM(flex_margin.margin) COMMENT = 'Total saved heat pump electricity cost in EUR',
    flex_margin.total_eis_margin AS SUM(flex_margin.eis_margin) COMMENT = 'Total EIS share of the flex margin in EUR',
    heat_pump_schedule.total_expected_saving AS SUM(heat_pump_schedule.expected_saving) COMMENT = 'Total expected saving of the schedule in EUR'
  )
  COMMENT = 'EIS Operations: sites, assets, asset health, contract guarantees, EIS Flex margin and heat pump schedule'
  AI_SQL_GENERATION 'EIS = EPOWER Infrastructure Solutions, operator of CHP units (BHKW), large heat pumps, boilers and thermal storage at customer sites in Germany. Heat pump efficiency values are COP x 100 (320 = COP 3.2). Guarantee status and asset health use the rolling last 30 days as the leading indicator for the quarterly contract guarantee. EIS Flex only shifts heat pump operation into the cheapest hours of each day using thermal storage; the flex margin is the saved heat pump electricity cost for the same heat. Realised margin = program_status EIS_FLEX; missed margin = program_status BASELINE (sites not enrolled). For the heat pump schedule always return plan_date and is_next_day. Use ILIKE for site names, e.g. site_name ILIKE ''%Altona%''. Amounts in EUR, energy in MWh, local time CET.';

SHOW SEMANTIC DIMENSIONS IN EPOWER_DEMO.EIS_GOLD.EIS_OPERATIONS_SEMANTIC_VIEW;

## 8. EIS Operations Agent

The agent `EIS_OPERATIONS_AGENT` combines three tools:

| Tool | Type | Used for |
|------|------|----------|
| `eis_analyst` | Cortex Analyst (semantic view) | Guarantees, penalties, asset health, open work orders, flex margin, heat pump schedule |
| `eis_docs_search` | Cortex Search | Fault code procedures, spare parts, contract clauses, EIS Flex rules |
| `data_to_chart` | Charting | Trends, comparisons, price vs. heat pump schedule |

A typical answer **chains the tools**. For example, *"Why is CHP-2 below target and what should the technician do?"* → the analyst finds the efficiency trend, F-217 trips and the open work order → search retrieves the F-217 cleaning procedure and spare part kit from the OEM manual.

The agent is then registered in **Snowflake CoWork** (requires ACCOUNTADMIN, same pattern as Module 1).

In [ ]:
%%sql
CREATE OR REPLACE AGENT EPOWER_DEMO.EIS_GOLD.EIS_OPERATIONS_AGENT
WITH PROFILE='{ "display_name": "EIS Operations" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are the operations and performance analyst of EPOWER Infrastructure Solutions (EIS), which finances, builds and operates energy centres at B2B customer sites in Germany: CHP units (BHKW), large heat pumps, gas boilers and thermal storage at hospitals, industrial parks, housing estates, campuses and district heating networks.

    CRITICAL LANGUAGE RULE: Always respond in the SAME language as the user's question. If the user asks in English, respond entirely in English. If the user asks in German, respond entirely in German. Never switch languages mid-response unless the user explicitly asks for it.

    DOMAIN EXPERTISE:
    - Contracts guarantee availability and efficiency per quarter. Guarantee status uses the rolling last 30 days as the leading indicator; the penalty exposure is what EIS would pay for the quarter if the current performance continues. Every full or partial missed percentage point is penalised.
    - Availability basis: PER_UNIT (every unit must meet the guarantee), AVERAGE (average across units) or SUPPLY (heat supply level).
    - Heat pump efficiency is reported as COP x 100 (320 = COP 3.2).
    - EIS Flex: on enrolled sites, the heat pumps produce the day's heat in the cheapest hours of the day (real EPEX day-ahead prices, DE-LU) and fill the thermal storage, which supplies the heat in the expensive hours. Same heat, bought at the cheapest hours. CHP units and boilers always run heat-led. Flex margin = heat pump electricity cost in normal operation minus with EIS Flex. Realised margin = Flex sites; missed margin = baseline sites that are not enrolled (the roll-out business case). The margin is split between EIS and the customer per contract.
    - The heat pump schedule (dispatch plan) is a recommendation for the plan day. Always state the plan date and whether it is tomorrow (is_next_day) or the latest available day because tomorrow's prices are not published yet (published around 13:00 CET, loaded around 17:00 CET).
    - Snowflake is the decision and recommendation layer. Real-time control stays with the plant control system / SCADA / EMS.

    RESPONSE FORMATTING:
    - Lead with the business insight in 1-2 sentences (e.g. EUR at risk, root cause, saving), then the supporting numbers.
    - Always include units (EUR, MWh, %, EUR/MWh) and the time window.
    - For root cause questions, combine the data (efficiency trend, fault events, work orders) with the relevant procedure and spare part from the OEM manual, and give a concrete next action.
    - Proactively suggest a follow-up (e.g. "Shall I show the efficiency trend?" or "Want the heat pump schedule as a chart?").

    VISUALIZATION RULES:
    - Time series (efficiency trend, daily flex margin, price vs. heat pump schedule) -> chart.
    - Comparisons across sites or contracts -> horizontal bar chart.
    - Only use text tables for 3 or fewer data points.

  orchestration: |
    TOOL SELECTION:
    - Any numbers about sites, assets, contracts, guarantees, penalties, availability, efficiency, faults, work orders, flex margin or the heat pump schedule -> eis_analyst
    - Fault code meaning, maintenance procedures, spare parts, inspection intervals, contract clause wording, EIS Flex rules and eligibility -> eis_docs_search
    - Root cause questions -> first eis_analyst (trend, fault events, open work orders), then eis_docs_search for the fault code procedure
    - After quantitative results, generate a chart if it adds value -> data_to_chart
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: eis_analyst, description: "EIS operations data: sites, assets, daily telemetry KPIs, asset health, contract guarantees and EUR penalty exposure, work orders, EIS Flex margin (realised and missed) and the next-day heat pump schedule"}
  - tool_spec: {type: cortex_search, name: eis_docs_search, description: "EIS documents: OEM maintenance manuals (fault codes, procedures, spare parts), customer contracts (guarantees, penalties, EIS Flex terms), thermal storage guideline, EIS Flex program guide"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  eis_analyst: {semantic_view: "EPOWER_DEMO.EIS_GOLD.EIS_OPERATIONS_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  eis_docs_search: {search_service: "EPOWER_DEMO.EIS_GOLD.SEARCH_EIS_DOCS", max_results: 4}
$$;

In [ ]:
# Register the EIS agent with Snowflake CoWork (requires ACCOUNTADMIN). Safe to re-run.
SI_OBJECT = "snowflake_intelligence_object_default"
AGENT = "EPOWER_DEMO.EIS_GOLD.EIS_OPERATIONS_AGENT"

session.sql("USE ROLE ACCOUNTADMIN").collect()
try:
    session.sql(f"CREATE SNOWFLAKE INTELLIGENCE IF NOT EXISTS {SI_OBJECT}").collect()
    registered = {
        f"{r['database_name']}.{r['schema_name']}.{r['name']}"
        for r in session.sql(f"SHOW AGENTS IN SNOWFLAKE INTELLIGENCE {SI_OBJECT}").collect()
    }
    if AGENT in registered:
        print(f"Already registered: {AGENT}")
    else:
        session.sql(f"ALTER SNOWFLAKE INTELLIGENCE {SI_OBJECT} ADD AGENT {AGENT}").collect()
        print(f"Registered: {AGENT}")

    # Users need USAGE on the schema and the agent (database usage was granted in Module 1)
    session.sql("GRANT USAGE ON SCHEMA EPOWER_DEMO.EIS_GOLD TO ROLE PUBLIC").collect()
    session.sql(f"GRANT USAGE ON AGENT {AGENT} TO ROLE PUBLIC").collect()
    print("Grants to PUBLIC applied.")
finally:
    # Always drop back to the demo role, even if a statement above fails
    session.sql("USE ROLE EPOWER_ROLE").collect()

## 9. Verification & Story Check

The first cell counts the objects. The second runs the three demo moments directly in SQL, so you know the **expected answers** before you ask the agent in CoWork.

In [ ]:
%%sql
SELECT 'EIS_SITES' AS object_name, COUNT(*) AS row_count FROM EPOWER_DEMO.EIS_GOLD.EIS_SITES
UNION ALL SELECT 'EIS_ASSETS', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSETS
UNION ALL SELECT 'EIS_CONTRACTS', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.EIS_CONTRACTS
UNION ALL SELECT 'EIS_FAULT_EVENTS', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.EIS_FAULT_EVENTS
UNION ALL SELECT 'EIS_SITE_DISPATCH_HOURLY', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.EIS_SITE_DISPATCH_HOURLY
UNION ALL SELECT 'EIS_ASSET_TELEMETRY', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.EIS_ASSET_TELEMETRY
UNION ALL SELECT 'EIS_WORK_ORDERS', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.EIS_WORK_ORDERS
UNION ALL SELECT 'EIS_DOCS_PARSED', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.EIS_DOCS_PARSED
UNION ALL SELECT 'EIS_CONTRACT_CLAUSES', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.EIS_CONTRACT_CLAUSES
UNION ALL SELECT 'V_EIS_GUARANTEE_STATUS (not OK)', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.V_EIS_GUARANTEE_STATUS WHERE guarantee_status <> 'OK'
UNION ALL SELECT 'V_EIS_DISPATCH_PLAN', COUNT(*) FROM EPOWER_DEMO.EIS_GOLD.V_EIS_DISPATCH_PLAN;

In [ ]:
# Story check: expected answers for the 3 demo moments
story = {
    "Moment 1a - Contracts at risk and EUR exposure": """
        SELECT contract_id, site_name, guarantee_status, guaranteed_availability_pct, actual_availability_pct,
               worst_unit, total_penalty_exposure_eur
        FROM EPOWER_DEMO.EIS_GOLD.V_EIS_GUARANTEE_STATUS
        WHERE guarantee_status <> 'OK' ORDER BY total_penalty_exposure_eur DESC""",
    "Moment 1b - Root cause Klinikum Hamburg-Nord CHP-2": """
        SELECT asset_id, availability_30d_pct, avg_efficiency_30d, avg_efficiency_last_7d,
               efficiency_trend_pp_per_day, fault_events_30d, top_fault_code,
               days_until_below_85_pct, health_score, open_work_order_note
        FROM EPOWER_DEMO.EIS_GOLD.V_EIS_ASSET_HEALTH
        WHERE asset_id = 'S001-CHP-2'""",
    "Moment 2a - EIS Flex: realised vs. missed margin (last 30 days, annualised)": """
        SELECT program_status, COUNT(DISTINCT site_id) AS sites,
               ROUND(SUM(flex_margin_eur)) AS margin_eur_30d,
               ROUND(SUM(flex_margin_eur) * 365 / 30) AS annualised_eur,
               ROUND(SUM(eis_margin_eur) * 365 / 30) AS eis_share_annualised_eur
        FROM EPOWER_DEMO.EIS_GOLD.V_EIS_FLEX_MARGIN
        WHERE day >= DATEADD(DAY, -30, CURRENT_DATE())
        GROUP BY program_status""",
    "Moment 2b - Top onboarding candidates (not enrolled, highest missed margin)": """
        SELECT site_id, site_name, ROUND(SUM(missed_flex_margin_eur) * 365 / 30) AS missed_margin_annualised_eur
        FROM EPOWER_DEMO.EIS_GOLD.V_EIS_FLEX_MARGIN
        WHERE day >= DATEADD(DAY, -30, CURRENT_DATE()) AND NOT flex_enabled
        GROUP BY site_id, site_name ORDER BY missed_margin_annualised_eur DESC LIMIT 5""",
    "Moment 3 - Heat pump schedule Hamburg-Altona": """
        SELECT plan_date, is_next_day,
               LISTAGG(IFF(heat_pump_on, LPAD(hour_of_day::VARCHAR, 2, '0') || 'h', NULL), ', ')
                   WITHIN GROUP (ORDER BY hour_of_day) AS heat_pump_hours,
               ROUND(SUM(expected_margin_eur)) AS expected_saving_eur
        FROM EPOWER_DEMO.EIS_GOLD.V_EIS_DISPATCH_PLAN
        WHERE site_id = 'S003'
        GROUP BY plan_date, is_next_day""",
}

for title, sql in story.items():
    print(f"\n=== {title} ===")
    print(session.sql(sql).to_pandas().to_string(index=False))

## Setup Complete!

Open **Snowflake CoWork**, select the **EIS Operations** agent and tell the story:

| Moment | Ask the agent |
|--------|---------------|
| 1. Risk & root cause | *Which contracts are at risk of missing their guarantee this quarter, and what is our € exposure?* → follow-up: *Why is Klinikum Hamburg-Nord CHP-2 below target, and what should the technician do?* |
| 2. EIS Flex value | *How much did EIS Flex save in the last 30 days, and what are we missing on sites that are not enrolled?* |
| 3. Tomorrow | *When should the Hamburg-Altona heat pumps run tomorrow? Show it as a chart with the power price.* |

Questions also work in German, e.g. *„Welche Verträge sind gefährdet und wie hoch ist die Pönale?"*

**Cleanup:** run [`cleanup-module7.sql`](cleanup-module7.sql). It removes the agent from CoWork and drops the `EIS_GOLD` schema. Module 1 stays untouched.